In [1]:
import re
import math
import pandas as pd
import synapse.ml.spark.aifunc as aifunc
import pyspark.sql.functions as F

from typing import Any
from pyspark.sql.functions import col, isnan, when, count, split, element_at, count_distinct, lower, desc, asc
from pyspark.sql.types import BooleanType, StringType, StructField, StructType, TimestampType, MapType

import time
import json
import asyncio
import aiohttp
from datetime import datetime, timezone
from pathlib import Path
from tenacity import AsyncRetrying, retry_if_exception_type, stop_after_attempt, wait_exponential


StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 3, Finished, Available, Finished, False)

Run this code below if poi_vi and poi_en haven't been created


# 1 Data Loading

In [2]:
poi_df = spark.sql(f"SELECT * FROM ai_poi")
display(poi_df.head(10))
print((poi_df.count(), len(poi_df.columns)))

StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7f008dc6-84f6-40fd-a0e5-8ec2ad458643)

(24308, 107)


In [3]:
poi_rating_df = spark.sql("SELECT * FROM ai_poi_rating")
poi_review_df = spark.sql("SELECT * FROM ai_poi_review")

StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 5, Finished, Available, Finished, False)

# 3 Configs

In [4]:
# Dry-run configs
DRY_RUN_MODE = False
DRY_RUN_SAMPLE_SIZE = 20

# Run mode:
#   'incremental' — only enrich POIs that are NOT already in the enrichment target table
#                   (the existing enriched POI list is fetched and skipped).
#   'backfill'    — (re-)enrich every scoped POI, even if it was enriched before.
RUN_MODE = 'incremental'

# Run scope (independent axis from DRY_RUN_MODE):
#   'full' — process the full scoped set (province + destination + ward filters apply).
#   'test' — additionally restrict to TEST_POI_IDS below. Exercises the full pipeline
#            (sibling minting → table writes → enrichment → ai_enriched_poi merge) on a
#            handful of POIs. DRY_RUN_MODE still controls *where* results are written;
#            RUN_SCOPE controls *which* POIs are processed.
#   'partner_portal - using the partner_portal only
RUN_SCOPE = 'partner_portal'
TEST_POI_IDS = [

]

# Pipeline configs
BATCH_SIZE = 40
MAX_RETRIES = 4
MAX_TOKENS = 900
DESCRIPTION_MAX_LEN = 200
REVIEW_TEXT_LIMIT = 350
EXPERIENCE_TITLE_MAX_LEN = 50
EXPERIENCE_DESCRIPTION_MAX_LEN = 150
# Multiple destinations supported. Add or remove IDs to expand scope.

PROVINCE_FILTER_NAME= ['Đà Nẵng', 'Huế']
# Legacy single-ID kept as alias for any old code that still references it.
CONCURRENCY_LIMIT = 8

# Ward selection — uncomment ONE line to choose the production batch
#selected_batch = ['An Hải', 'Hòa Cường']
#selected_batch = ['An Hải', 'Hòa Cường']  # Batch 1 — 6,307 POIs
selected_batch = []  # Leave it empty for no filtering
# selected_batch = ['Ngũ Hành Sơn', 'Cẩm Lệ', 'Hòa Khánh', 'Hải Vân', 'Liên Chiểu', 'An Khê', 'Hội An', 'Hội An Tây', 'Hòa Xuân', 'Bà Nà', 'Hội An Đông', 'An Thắng', 'Hòa Vang']  # Batch 3 — 2,753 POIs
# Dry-run ward selection — used when DRY_RUN_MODE = True (pick a few small wards to smoke-test)
selected_dryrun_batch = ['Sơn Trà'] # not
# selected_dryrun_batch = ['An Hải']
# selected_dryrun_batch = ['Hòa Khánh', 'Hải Vân']

# POIs ingested from the partner portal must be enriched in EVERY ward batch — the ward
# filter (Filtering cell) never drops them. Change this if ai_poi.ingestion_source uses a
# different literal for partner-portal rows (matched case-insensitively, whitespace-trimmed).
PARTNER_PORTAL_SOURCE = 'partner_portal'

SUPPORTED_LANGS = ['vi', 'en', 'ko']
DEFAULT_LANG = 'vi'
LANGUAGE_NAMES = {
    'vi': 'Vietnamese',
    'en': 'English',
    'ko': 'Korean',
}

# Fabric source tables
LAKEHOUSE_NAME = 'll_vv_gold'
POI_TABLE = 'ai_poi'

# Silver tables we read source rows from AND write sibling rows into when minting the
# missing-language sibling. See docs/en_poi_generation_design.md §3.
POI_MASTER_TABLE = 'poi_master'
POI_DETAIL_TABLE = 'poi_detail'
POI_RATING_TABLE = 'poi_rating'
POI_REVIEW_TABLE = 'poi_review'

# Gold merge target written at the end of this notebook (§3.7 of the design).
AI_ENRICHED_POI_TABLE = 'ai_enriched_poi'

# Set to False to skip writing the *core* tables (poi_master/poi_detail/poi_rating/
# poi_review/ai_poi) during sibling creation — useful while iterating on the prompts.
# The enrichment side-tables still get written. Independent of DRY_RUN_MODE.
WRITE_SIBLING_CORE_TABLES = True

# Fabric target tables
WRITE_MODE = 'test_table' if DRY_RUN_MODE else 'side_table'

DRY_RUN_POI_TABLE = 'poi_enrichment_dryrun'
DRY_RUN_POI_DISPLAY_TABLE = 'poi_enrichment_display_dryrun'
DRY_RUN_DEST_TABLE = 'destination_enrichment_dryrun'
DRY_RUN_DEST_DISPLAY_TABLE = 'destination_enrichment_display_dryrun'
DRY_RUN_AI_ENRICHED_POI_TABLE = 'ai_enriched_poi_dryrun'

POI_ENRICHMENT_TABLE = 'poi_enrichment'
POI_ENRICHMENT_DISPLAY_TABLE = 'poi_enrichment_display'
DEST_ENRICHMENT_TABLE = 'destination_enrichment'
DEST_ENRICHMENT_DISPLAY_TABLE = 'destination_enrichment_display'

TARGET_POI_OUTPUT = DRY_RUN_POI_TABLE if DRY_RUN_MODE else POI_ENRICHMENT_TABLE
TARGET_POI_DISPLAY_OUTPUT = DRY_RUN_POI_DISPLAY_TABLE if DRY_RUN_MODE else POI_ENRICHMENT_DISPLAY_TABLE
TARGET_DEST_OUTPUT = DRY_RUN_DEST_TABLE if DRY_RUN_MODE else DEST_ENRICHMENT_TABLE
TARGET_DEST_DISPLAY_OUTPUT = DRY_RUN_DEST_DISPLAY_TABLE if DRY_RUN_MODE else DEST_ENRICHMENT_DISPLAY_TABLE
# ai_enriched_poi (gold merge target, §4.7) — dry-run writes to a disposable sibling table.
TARGET_AI_ENRICHED_POI_OUTPUT = DRY_RUN_AI_ENRICHED_POI_TABLE if DRY_RUN_MODE else AI_ENRICHED_POI_TABLE

# Staging table for raw run_enrichment() output — overwritten each batch so downstream cells can resume without re-running the LLM
POI_ENRICHMENT_STAGING_TABLE = 'poi_enrichment_staging_dryrun' if DRY_RUN_MODE else 'poi_enrichment_staging'


# Per-destination prompt context. The enrichment LLM picks the right one for each POI
# via row.get('destination_id') in build_poi_payload below. Add new entries as you onboard
# more destinations; keep the structure consistent so the model parses it the same way.

GENERIC_VIETNAM_CONTEXT = """
Vietnam is a long, narrow country with three broad climate regions (north, central, south).
- Currency: Vietnamese Dong (VND)
- Most coastal areas have a dry season and a wet/rainy season — check the destination's local
  seasonal notes when seasonality matters for outdoor experiences.
"""

DA_NANG_CONTEXT = """
Da Nang is a coastal city in central Vietnam, known for its beaches, mountains, and proximity to UNESCO World Heritage Sites.
- Dry season: February to August
- Rainy season: September to January
- Currency: Vietnamese Dong (VND)
- Major visitor zones include My Khe Beach, Ba Na Hills, Son Tra Peninsula (Monkey Mountain), Han River District, and the nearby ancient town of Hoi An
- Outdoor and beach experiences are best from March to August
- Notable landmarks include the Dragon Bridge, Marble Mountains, and the Golden Bridge (Ba Na Hills)
"""

PHU_QUOC_CONTEXT = """
Phu Quoc is a tropical island in southern Vietnam in Kien Giang province.
- Dry season: November to April
- Rainy season: May to October
- Currency: Vietnamese Dong (VND)
- Major visitor zones include Duong Dong, Long Beach, An Thoi, Sao Beach, and Phu Quoc National Park
- Outdoor and sea-based experiences are usually best in dry season
"""

BA_NA_HILLS_CONTEXT = """
Ba Na Hills is a mountain resort at 1,487 m elevation in Da Nang, central Vietnam, accessed by a record-holding cable car system from a base station about 30 km west of Da Nang city.
- Cool, alpine climate year-round (typical daytime temperatures 15-22°C); the summit is often misty or in cloud
- Dry season at altitude: roughly February to August; wetter and foggier September to January
- Currency: Vietnamese Dong (VND)
- Iconic landmarks: the Golden Bridge held up by two giant stone hands, the French Village (a re-creation of a French alpine town), Linh Ung Pagoda, Le Jardin d'Amour themed gardens, Debay Wine Cellar
- Indoor entertainment: Fantasy Park (one of Asia's largest indoor amusement complexes)
- Most visitors come on a day trip from Da Nang; same-day return is the norm but on-site hotels exist
- Outdoor garden / bridge photo experiences are best March to September on clear days; bring a light jacket year-round
"""

HOI_AN_CONTEXT = """
Hoi An is a UNESCO World Heritage Ancient Town on the Thu Bon River in central Vietnam (Quang Nam province), about 30 km south of Da Nang.
- Dry season: February to August
- Flood-prone season: October to November (the Thu Bon River regularly rises into the Ancient Town)
- Currency: Vietnamese Dong (VND)
- The Ancient Town is pedestrian-friendly during the day and lantern-lit at night; key landmarks include the Japanese Covered Bridge, Tan Ky Old House, Phuc Kien (Fujian) Assembly Hall, Trieu Chau Assembly Hall, and the Central Market
- Famous for silk lanterns, custom tailoring, leather workshops, and night markets
- Nearby beaches: An Bang and Cua Dai (about 10-15 minutes by bicycle); the Cham Islands snorkeling/diving trips are accessible by speedboat in calm weather
- Signature local foods: cao lau noodles, white rose dumplings (banh bao banh vac), com ga Hoi An (chicken rice), banh mi (notably Banh Mi Phuong and Madam Khanh)
- The Lantern Full Moon Festival happens on the 14th day of each lunar month — motor traffic is banned in the Ancient Town
- Outdoor and beach experiences are best March to August
"""

LANG_CO_CONTEXT = """
Lang Co is a coastal resort town in Thua Thien Hue province at the foot of Hai Van Pass, roughly halfway between Hue and Da Nang.
- Dry season: March to August
- Rainy season: September to February (occasional typhoons)
- Currency: Vietnamese Dong (VND)
- About 10 km of white-sand beach along Lang Co Bay; the sheltered Lap An (Lang Co) lagoon sits between the beach and the mountains
- Famous for fresh seafood, especially Lang Co lagoon oysters (hau Lang Co), squid, and lagoon fish
- Home to luxury beach resorts (Banyan Tree Lang Co, Angsana Lang Co) and Laguna Lang Co golf course
- Quick connection to both Hue (north) and Da Nang (south) via Hai Van Pass (scenic coastal road) or Hai Van Tunnel (faster)
- Outdoor and beach experiences are best April to August
"""

HUE_CONTEXT = """
Hue is the former imperial capital of the Nguyen Dynasty (1802-1945), in central Vietnam on the Perfume River (Song Huong).
- Dry season: March to August
- Rainy season: September to February (typhoons possible; floods occasionally affect riverside districts)
- Currency: Vietnamese Dong (VND)
- The UNESCO-listed Complex of Hue Monuments includes the Imperial Citadel (Hoang Thanh), the Forbidden Purple City (Tu Cam Thanh), and the royal tombs of emperors Tu Duc, Khai Dinh, Minh Mang, and Gia Long
- Notable spiritual sites: Thien Mu Pagoda on the Perfume River, Tu Hieu Pagoda, Bach Ma National Park nearby
- Sophisticated imperial cuisine: com hen (baby clam rice), bun bo Hue (spicy beef noodle), banh khoai, banh beo, banh nam, banh loc — generally smaller portions, more delicate spicing than other regional cuisines
- Dragon-boat cruises on the Perfume River and royal court music (Nha Nhac) performances
- The Hue Festival is a biennial cultural festival held in even-numbered years
- Outdoor sightseeing of citadels and tombs is best March to August
"""

# Lookup by destination_id. Add new entries as more destinations come online.
DESTINATION_CONTEXTS = {
    'qen7pnt2uffq9pz2q6b59k1n': BA_NA_HILLS_CONTEXT,
    'jqho2nmaug9cojnc3r4p7tll': HOI_AN_CONTEXT,
    'fh2ik0okgqebb0cfcds25e46': HUE_CONTEXT,
    'nwzsg28kmk8h4xkai5udjwy7': DA_NANG_CONTEXT,
    'kmro0kny8au4eodf3l4p19yk': PHU_QUOC_CONTEXT,
}

# Azure AI Foundry
AZURE_AI_ENDPOINT = 'https://devagenticplatformapi.openai.azure.com'
AZURE_AI_DEPLOYMENT = 'gpt-4.1'
AZURE_AI_API_VERSION = '2024-12-01-preview'
AZURE_AI_API_KEY = 'REDACTED'

REF_TYPE_BY_SECTOR = {
    'accommodation': [
        'hotel', 'resort_hotel', 'hostel', 'motel', 'guest_house', 'campground',
    ],
    'attraction': [
        'theme_park', 'water_park', 'museum', 'historical_place', 'zoo_and_aquarium',
        'national_park', 'landmark', 'eco_tourism', 'art_show_performance',
        'traditional_village', 'religious_site', 'casino', 'art_gallery',
    ],
    'event': [
        'music_entertainment_event', 'sports_event', 'education_business_event',
        'festival_community_event', 'arts_culture_event',
    ],
    'f_n_b': [
        'vietnamese_restaurant', 'chinese_restaurant', 'japanese_restaurant',
        'korean_restaurant', 'thai_restaurant', 'french_restaurant', 'italian_restaurant',
        'american_restaurant', 'western_restaurant', 'internation_restaurant',
        'fusion_restaurant', 'cafe', 'bakery_dessert', 'night_club', 'fast_food',
        'vegetarian_restaurant', 'seafood_restaurant',
    ],
    'shopping_n_retail': [
        'local_specialty', 'souvenir_handicraft', 'fashion_accessories', 'jewelry_store',
        'cosmetics_store', 'grocery_store', 'furniture_store', 'book_store',
        'electronics_store', 'shopping_mall', 'convenience_store', 'local_market',
        'sporting_goods_store',
    ],
    'spa_n_wellness': [
        'spa', 'therapeutic_massage', 'clinic_aesthetic_spa', 'blind_massage', 'onsen',
        'hair_salon', 'beauty_salon', 'fitness_center', 'yoga_pilates_studio',
        'meditation_center',
    ],
    'tour_n_activities': [
        'travel_agency',
    ],
    'transport_on_demand': [
        'taxi_service', 'chauffeur_charter_service', 'ride_hailing_platform',
        'airport_limousine_service',
    ],
    'transport_route_base': [
        'sleeper_bus_service', 'limousine_service', 'speedboat_service',
        'ferry_service', 'train_service',
    ],
    'vehicle_rental': [
        'motocycle_rental_agency', 'self_drive_car_rental', 'chauffeur_car_rental',
        'bicycle_ev_rental', 'campervan_rental',
    ],
    # currently set this is "other" to match with B2B
    'other': [
        # Transit & Infrastructure
        'airport', 'bus_station', 'train_station', 'metro_station',
        'ferry_terminal', 'transit_station',
        # Nature & Outdoors
        'park', 'natural_feature', 'town_square',
        # Government & Public Services
        'government_offices', 'embassy', 'police_and_fire_station',
        'post_office', 'public_library',
        # Education & Healthcare
        'hospital', 'drugstore', 'academic_building',
        # Religious Places
        'public_religious_site',
    ],
    'other': [],
}
ALL_REF_TYPES = sorted({t for ts in REF_TYPE_BY_SECTOR.values() for t in ts})


StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 6, Finished, Available, Finished, False)

## 4.1. Preprocessing

In [5]:
# =========== Helper constants ===========
CATEGORY_GROUPS = {
    'Lodging': {'lodging', 'hotel', 'resort_hotel', 'motel', 'guest_house', 'hostel', 'bed_and_breakfast', 'resort'},
    'Main Meals': {'restaurant', 'food', 'meal_takeaway', 'seafood_restaurant', 'bistro'},
    'Light Snacks / Cafe': {'cafe', 'bakery', 'coffee_shop', 'tea_house'},
    'Heavy Entertainment': {'amusement_park', 'theme_park', 'water_park', 'zoo', 'aquarium', 'national_park', 'hiking_area'},
    'Light Sightseeing / Culture & Nature': {'tourist_attraction', 'museum', 'place_of_worship', 'beach', 'park', 'historical_landmark', 'pagoda', 'church'},
    'Shopping & Nightlife': {'shopping_mall', 'market', 'night_market', 'bar', 'night_club', 'souvenir_store'},
}

DURATION_PRIORS = {
    'Lodging': (1200, 1200),
    'Main Meals': (60, 90),
    'Light Snacks / Cafe': (45, 60),
    'Heavy Entertainment': (180, 360),
    'Light Sightseeing / Culture & Nature': (60, 120),
    'Shopping & Nightlife': (60, 180),
    'Other': (60, 90),
}

OUTDOOR_TYPES = {'beach', 'park', 'tourist_attraction', 'hiking_area', 'national_park', 'amusement_park', 'water_park', 'campground'}
BEST_TIME_MONTH_RANGES = {
    'all': {'type': 'all_year', 'from_month': None, 'to_month': None},
    'dry_season': {'type': 'month_range', 'from_month': 11, 'to_month': 4},
    'rainy_season': {'type': 'month_range', 'from_month': 5, 'to_month': 10},
    'beach_coast': {'type': 'month_range', 'from_month': 12, 'to_month': 4},
    'island_boat': {'type': 'month_range', 'from_month': 11, 'to_month': 4},
    'snorkeling_diving': {'type': 'month_range', 'from_month': 12, 'to_month': 5},
    'waterfall_green_season': {'type': 'month_range', 'from_month': 6, 'to_month': 10},
    'forest_hiking': {'type': 'month_range', 'from_month': 11, 'to_month': 3},
    'outdoor_entertainment': {'type': 'month_range', 'from_month': 11, 'to_month': 4},
    'indoor_culture': {'type': 'all_year', 'from_month': None, 'to_month': None},
    'shopping_nightlife': {'type': 'all_year', 'from_month': None, 'to_month': None},
    'spa_wellness': {'type': 'all_year', 'from_month': None, 'to_month': None},
}
EXPERIENCE_KEYWORDS = {
    'wb_twilight','travel_explore','temple_buddhist',
    'spa','shopping_bag','scuba_diving','sailing',
    'ramen_dining','paragliding','paid','local_bar',
    'kayaking','hiking','hand_package','directions_bus',
    'directions_boat','dinner_dining','curtains','concierge',
    'coffee','car_rental','beer_meal',
    'attractions','pending'
}

# Default keyword used when the model returns a value outside EXPERIENCE_KEYWORDS.
EXPERIENCE_KEYWORD_FALLBACK = 'pending'

TRUE_FLAG_COLUMNS = [
    'wifi_free', 'has_swimming_pool', 'has_gym', 'has_spa', 'has_bar_on_site',
    'has_restaurant_on_site', 'has_garden', 'has_sauna', 'has_jacuzzi',
    'has_meeting_rooms', 'has_coworking_space', 'has_airport_shuttle',
    'front_desk_24h', 'luggage_storage', 'currency_exchange', 'ev_charge',
    'braille_elevator', 'lowered_sink', 'air_conditioning', 'has_restroom',
    'accessibility_options_wheelchair_accessible_parking',
    'accessibility_options_wheelchair_accessible_entrance',
    'accessibility_options_wheelchair_accessible_restroom',
    'parking', 'payment_options_accepts_credit_cards',
    'payment_options_accepts_debit_cards', 'payment_options_accepts_nfc',
    'outdoor_seating', 'live_music', 'takeout', 'delivery', 'dine_in',
    'curbside_pickup', 'reservable', 'serves_breakfast', 'serves_lunch',
    'serves_dinner', 'serves_beer', 'serves_wine', 'serves_brunch',
    'serves_cocktails', 'serves_vegetarian_food', 'serves_dessert',
    'serves_coffee', 'menu_for_children', 'good_for_children', 'allows_dogs',
    'good_for_groups', 'good_for_watching_sports', 'pure_service_area_business',
    'free_cancellation'
]

review_cols = [
    'poi_id', 'review_id', 'review_date', 'updated_at',
    'rating', 'title', 'review_text', 'positive', 'negative', 'response_text',
]

# =========== Helper functions ===========

def is_nullish(value):
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if isinstance(value, str) and value.strip() == '':
        return True
    return False


def clip_text(text, limit):
    if is_nullish(text):
        return ''
    text = re.sub(r'\s+', ' ', str(text)).strip()
    if len(text) <= limit:
        return text
    return text[: limit - 3].rstrip() + '...'


def as_list(value):
    """Parse a comma-separated string or JSON array string into a plain Python list."""
    if is_nullish(value):
        return []
    if isinstance(value, (list, tuple)):
        return list(value)
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        if value.startswith('['):
            try:
                parsed = json.loads(value)
                return parsed if isinstance(parsed, list) else [parsed]
            except Exception:
                pass
        return [part.strip() for part in value.split(',') if part.strip()]
    return [value]

def infer_group(primary_type, types):
    candidates = {primary_type, *types}
    for group_name, group_types in CATEGORY_GROUPS.items():
        if candidates & group_types:
            return group_name
    return 'Other'


def infer_duration_bounds(primary_type, types, name):
    low, high = DURATION_PRIORS[infer_group(primary_type, types)]
    tokens = (name + ' ' + primary_type + ' ' + ' '.join(types)).lower()
    if any(term in tokens for term in ['cable car', 'safari', 'theme', 'vinwonders', 'water park']):
        return (180, 360)
    if any(term in tokens for term in ['night market', 'market']):
        return (90, 150)
    if any(term in tokens for term in ['pagoda', 'temple', 'prison', 'museum']):
        return (60, 120)
    if any(term in tokens for term in ['beach', 'snorkel', 'island', 'boat']):
        return (120, 240)
    return (low, high)


def infer_best_time_code(primary_type, types, name):
    tokens = {primary_type, *types}
    haystack = (name + ' ' + primary_type + ' ' + ' '.join(types)).lower()

    if any(term in haystack for term in ['snorkel', 'snorkeling', 'diving', 'scuba']):
        return 'snorkeling_diving'
    if any(term in haystack for term in ['island', 'boat', 'canoe', 'ferry', 'sailing']):
        return 'island_boat'
    if any(term in haystack for term in ['beach', 'sunset', 'coast', 'coastal']):
        return 'beach_coast'
    if any(term in haystack for term in ['waterfall', 'stream', 'suối', 'thác']):
        return 'waterfall_green_season'
    if tokens & {'national_park', 'hiking_area'} or any(term in haystack for term in ['forest', 'hiking', 'trekking', 'trail']):
        return 'forest_hiking'
    if tokens & {'museum', 'place_of_worship'} or any(term in haystack for term in ['museum', 'pagoda', 'temple', 'prison', 'church']):
        return 'indoor_culture'
    if tokens & {'shopping_mall', 'market', 'night_market'} or any(term in haystack for term in ['market', 'shopping', 'mall', 'night market']):
        return 'shopping_nightlife'
    if any(term in haystack for term in ['spa', 'wellness', 'massage']):
        return 'spa_wellness'
    if tokens & {'amusement_park', 'theme_park', 'water_park'}:
        return 'outdoor_entertainment'
    if tokens & OUTDOOR_TYPES:
        return 'dry_season'
    return 'all'

def safe_month(value):
    try:
        month = int(value)
    except (TypeError, ValueError):
        return None
    return month if 1 <= month <= 12 else None


def best_time_from_code(best_time_code: str) -> dict[str, Any]:
    return dict(BEST_TIME_MONTH_RANGES.get(best_time_code, BEST_TIME_MONTH_RANGES['all']))


def normalize_best_time(value, fallback_code: str) -> dict[str, Any]:
    if isinstance(value, str):
        if value in BEST_TIME_MONTH_RANGES:
            return best_time_from_code(value)
        if value.strip().lower() == 'all':
            return best_time_from_code('all')

    if isinstance(value, dict):
        best_time_type = value.get('type')
        if best_time_type == 'all_year':
            return best_time_from_code('all')

        from_month = safe_month(value.get('from_month'))
        to_month = safe_month(value.get('to_month'))
        if best_time_type == 'month_range' and from_month is not None and to_month is not None:
            return {'type': 'month_range', 'from_month': from_month, 'to_month': to_month}

    return best_time_from_code(fallback_code)


def format_best_time_value(best_time: dict[str, Any]) -> str:
    if best_time.get('type') == 'all_year':
        return 'all'
    return f"{best_time['from_month']}-{best_time['to_month']}"


def fallback_experience(row: pd.Series, index: int) -> dict[str, str]:
    """Single-language fallback highlight (in the POI's own language) used when the model
    omits or malforms an experience."""
    lang = row.get('lang') if row.get('lang') in SUPPORTED_LANGS else DEFAULT_LANG
    short_type = row['primary_type'].replace('_', ' ')
    fallback_items_by_lang = {
        'vi': [
            {
                'title': f"Khám phá {short_type}",
                'description': f"Trải nghiệm nét đặc trưng của {short_type} trong lịch trình của bạn.",
                'keyword': 'travel_explore',
            },
            {
                'title': 'Dễ ghép lịch trình',
                'description': 'Một điểm dừng linh hoạt, phù hợp để kết hợp cùng các trải nghiệm gần đó.',
                'keyword': EXPERIENCE_KEYWORD_FALLBACK,
            },
            {
                'title': 'Gợi ý trước chuyến đi',
                'description': 'Nên kiểm tra đánh giá mới và thông tin mở cửa trước khi lên đường.',
                'keyword': EXPERIENCE_KEYWORD_FALLBACK,
            },
        ],
        'en': [
            {
                'title': f"Explore the {short_type}",
                'description': f"Experience what makes this {short_type} distinctive on your trip.",
                'keyword': 'travel_explore',
            },
            {
                'title': 'Easy to fit in',
                'description': 'A flexible stop that pairs well with nearby experiences.',
                'keyword': EXPERIENCE_KEYWORD_FALLBACK,
            },
            {
                'title': 'Before you go',
                'description': 'Check recent reviews and opening hours before you set out.',
                'keyword': EXPERIENCE_KEYWORD_FALLBACK,
            },
        ],
        'ko': [
            {
                'title': f"{short_type} 둘러보기",
                'description': f"여행 일정에서 이 {short_type}만의 매력을 경험해 보세요.",
                'keyword': 'travel_explore',
            },
            {
                'title': '일정에 넣기 좋은 장소',
                'description': '주변 경험과 함께 묶기 좋은 유연한 방문지입니다.',
                'keyword': EXPERIENCE_KEYWORD_FALLBACK,
            },
            {
                'title': '방문 전 확인',
                'description': '출발 전에 최신 후기와 운영 시간을 확인하는 것이 좋습니다.',
                'keyword': EXPERIENCE_KEYWORD_FALLBACK,
            },
        ],
    }
    return fallback_items_by_lang[lang][index]


def normalize_allowed_string(value, allowed_values: set[str], fallback: str, lowercase: bool = False) -> str:
    """Return a validated string value without attempting set membership on unhashable model output."""
    if isinstance(value, str):
        candidate = value.strip()
        if lowercase:
            candidate = candidate.lower()
        if candidate in allowed_values:
            return candidate
    return fallback


def normalize_experience_item(item, row: pd.Series, index: int) -> dict[str, str]:
    """Return one highlight as a single-language dict (in the POI's own language)."""
    fb = fallback_experience(row, index)
    if isinstance(item, dict):
        title = clip_text(
            item.get('title') or item.get('title_vi') or item.get('title_en') or item.get('title_ko') or fb['title'],
            EXPERIENCE_TITLE_MAX_LEN,
        )
        description = clip_text(
            item.get('description') or item.get('desc') or item.get('description_vi') or item.get('description_en') or item.get('description_ko') or fb['description'],
            EXPERIENCE_DESCRIPTION_MAX_LEN,
        )
        keyword = normalize_allowed_string(item.get('keyword'), EXPERIENCE_KEYWORDS, fb['keyword'])
    elif not is_nullish(item):
        text = clip_text(item, EXPERIENCE_DESCRIPTION_MAX_LEN)
        title = clip_text(text, EXPERIENCE_TITLE_MAX_LEN)
        description = text
        keyword = fb['keyword']
    else:
        title = fb['title']
        description = fb['description']
        keyword = fb['keyword']
    keyword = f"material_{keyword}"
    return {'title': title, 'description': description, 'keyword': keyword}


def normalize_experiences(value, row: pd.Series) -> list[dict[str, str]]:
    """Normalize raw model highlights into a single list of exactly 3 items in the POI's language."""
    raw_items = as_list(value)
    items = [normalize_experience_item(item, row, idx) for idx, item in enumerate(raw_items[:3])]
    while len(items) < 3:
        items.append(normalize_experience_item(None, row, len(items)))
    return items[:3]

def safe_int(value):
    if is_nullish(value):
        return None
    try:
        return int(float(value))
    except Exception:
        return None


def observed_cost_vnd(row):
    low = safe_int(row.get('price_min'))
    high = safe_int(row.get('price_max'))
    if low is not None and high is not None:
        return int(round((low + high) / 2))
    return None


def round_vnd(value):
    return int(round(float(value) / 1000.0) * 1000)


def build_true_flags(row):
    """Return the list of boolean service-flag column names that are True on this row."""
    return [col for col in TRUE_FLAG_COLUMNS if col in row and bool(row[col])]


def compact_review_text(row):
    parts = []
    rating = row.get('rating')
    if not is_nullish(rating):
        parts.append(f'Rating {rating}/5')
    for col in ['title', 'review_text', 'positive', 'negative', 'response_text']:
        val = row.get(col)
        if not is_nullish(val):
            label = col.replace('_', ' ').title()
            parts.append(f'{label}: {clip_text(val, REVIEW_TEXT_LIMIT)}')
    return clip_text(' | '.join(parts), REVIEW_TEXT_LIMIT)

def normalize_poi_dataframe(df):
    """Produce the working columns expected by downstream stages from the merged source DataFrame."""
    work = df.copy()

    # Empty-input short-circuit: pandas apply(axis=1) on an empty frame returns a
    # DataFrame (not a Series), which then fails when assigned to a single column.
    # Pre-populate every column the downstream stages expect so they see a clean empty
    # frame and can no-op cleanly instead of crashing.
    if work.empty:
        for col in ['name', 'primary_type', 'lang', 'types_norm', 'rating',
                    'user_rating_count', 'price_level', 'opening_hours_text',
                    'review_texts', 'zone_name', 'category_group',
                    'duration_prior_min', 'duration_prior_max', 'best_time_candidate',
                    'observed_cost_vnd', 'service_flags']:
            if col not in work.columns:
                work[col] = pd.Series(dtype='object')
        return work

    # Core identity — column names are guaranteed by the fixed schema
    work['name'] = work['poi_name']
    work['primary_type'] = work['business_category'].fillna('unknown').str.strip()

    # Per-POI content language
    work['lang'] = work['lang'].where(work['lang'].isin(SUPPORTED_LANGS), DEFAULT_LANG)

    # Merge poi_types and subcategory_tags into a single type list
    work['types_norm'] = work.apply(
        lambda row: as_list(row.get('poi_types')) + as_list(row.get('subcategory_tags')), axis=1
    )

    # Rating and review count come directly from poi.rating_overall and poi.rating_count
    # (aggregated by the ingestion pipeline from Google Places, single source of truth)
    work['rating'] = pd.to_numeric(work['rating_overall'], errors='coerce')
    work['user_rating_count'] = pd.to_numeric(work['rating_count'].fillna(0), errors='coerce')

    work['price_level'] = pd.to_numeric(work['price_level'], errors='coerce')

    # operating_hours is a plain string in the schema; wrap in a list for downstream consumers
    work['opening_hours_text'] = work['operating_hours'].apply(
        lambda val: [clip_text(str(val), 200)] if not is_nullish(val) else []
    )

    work['review_texts'] = work['review_texts'].apply(lambda items: items if isinstance(items, list) else [])
    #work['destination_name'] = work['destination_name'].fillna('Phu Quoc')
    work['zone_name'] = work['district'].fillna(work['destination_name'])
    work['category_group'] = work.apply(lambda row: infer_group(row['primary_type'], row['types_norm']), axis=1)

    duration_bounds = work.apply(
        lambda row: infer_duration_bounds(row['primary_type'], row['types_norm'], row['name']), axis=1
    )
    work['duration_prior_min'] = duration_bounds.apply(lambda t: t[0])
    work['duration_prior_max'] = duration_bounds.apply(lambda t: t[1])

    work['best_time_candidate'] = work.apply(
        lambda row: infer_best_time_code(row['primary_type'], row['types_norm'], row['name']), axis=1
    )
    work['observed_cost_vnd'] = work.apply(observed_cost_vnd, axis=1)
    work['service_flags'] = work.apply(build_true_flags, axis=1)
    return work


StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 7, Finished, Available, Finished, False)

In [6]:
# Filtering
# Partner-portal POIs must ALWAYS be enriched: they bypass every scope filter in this cell
# (is_active, province, ward). Everything else must pass the normal scope filters.
_from_partner_portal = F.lower(F.trim(F.col('ingestion_source'))) == PARTNER_PORTAL_SOURCE



# Ward filter: keep only POIs whose ward is in the selected batch. An empty ward list means
# 'no ward restriction' so the filter is skipped. Partner-portal POIs are kept regardless of ward.
ward_filter = selected_dryrun_batch if DRY_RUN_MODE else selected_batch
if len(ward_filter) > 0:
    poi_df = poi_df.filter(F.col('ward').isin(ward_filter) | _from_partner_portal)
    print(f'Ward filter ({"dryrun" if DRY_RUN_MODE else "prod"}): {ward_filter} '
          f'(+ all ingestion_source="{PARTNER_PORTAL_SOURCE}" POIs kept regardless of ward)')
else:
    print(f'Ward filter ({"dryrun" if DRY_RUN_MODE else "prod"}): none (empty ward list) — keeping all wards')


# RUN_SCOPE filter (cell 17 configs). In 'test' mode we restrict to TEST_POI_IDS so the
# full pipeline runs against a handful of POIs end-to-end. An empty TEST_POI_IDS in test
# mode raises rather than silently producing an empty run.
if RUN_SCOPE == 'test':
    print("RUN_SCOPE='test': processing the selected set")
    if not TEST_POI_IDS:
        raise ValueError(
            "RUN_SCOPE='test' but TEST_POI_IDS is empty. Populate TEST_POI_IDS in the configs "
            "cell, or switch RUN_SCOPE to 'full'."
        )
    poi_df = poi_df.filter(F.col('poi_id').isin(TEST_POI_IDS))
    print(f"RUN_SCOPE='test': restricted to {len(TEST_POI_IDS)} poi_id(s)")

elif RUN_SCOPE == 'partner_portal':
    print("RUN_SCOPE='partner portal': processing the business_service only")
    poi_df = poi_df.filter(
        (F.col('ingestion_source') == 'partner_portal') &
        F.col('destination_id').isNotNull() # MUST HAVE THIS FOR QUALITY DATA
    )
else:
    poi_df = poi_df.filter(
    (F.col('is_active') & F.col('province_name').isin(PROVINCE_FILTER_NAME)) | _from_partner_portal)
    print("RUN_SCOPE='full': processing the full scoped set")

# Sanity gate: the scoped POI must ALSO exist in poi_master. Without a master row we
# can't build a sibling poi_master record, and the downstream cells would silently skip
# those rows after a warning. Keep only the intersection here so the filter is explicit.
_scoped_before_master = poi_df.select('poi_id').toPandas()['poi_id'].astype(str).tolist()
print('before join with poi_master, total:',len(_scoped_before_master))
_master_ids_df = spark.table(POI_MASTER_TABLE).select('poi_id').distinct()
poi_df = poi_df.join(_master_ids_df, on='poi_id', how='inner')
_scoped_after_master = poi_df.select('poi_id').toPandas()['poi_id'].astype(str).tolist()
_dropped_missing_master = sorted(set(_scoped_before_master) - set(_scoped_after_master))
print(
    f'poi_master coverage filter: kept {len(_scoped_after_master)}/{len(_scoped_before_master)} '
    f'POIs present in both ai_poi and poi_master (dropped {len(_dropped_missing_master)})'
)
if _dropped_missing_master:
    print(f'  dropped poi_ids (not in poi_master): {_dropped_missing_master[:20]}'
          f'{"…" if len(_dropped_missing_master) > 20 else ""}')

# Run-mode summary (the incremental left-anti against poi_enrichment was moved out of
# this cell to *after* the sibling stage in §4.0.9 — that way sibling minting always
# happens regardless of RUN_MODE, and the incremental filter still avoids re-billing
# previously-enriched poi_ids).
if RUN_MODE == 'incremental':
    print('Incremental mode: already-enriched poi_ids will be filtered after the sibling stage (§4.0.9).')
else:
    print('Backfill mode: (re-)enriching all scoped POIs (incl. newly-minted siblings)')

if DRY_RUN_MODE:
    print("DRY RUN LIMIT")
    poi_df = poi_df.orderBy(desc('rating_count'), asc('poi_id')).limit(DRY_RUN_SAMPLE_SIZE)
    poi_rating_df = poi_rating_df.orderBy( asc('rating_id')).limit(DRY_RUN_SAMPLE_SIZE)
    poi_review_df = poi_review_df.orderBy( asc('poi_id')).limit(DRY_RUN_SAMPLE_SIZE)

scoped_poi_ids = poi_df.select('poi_id').toPandas()['poi_id'].tolist()
display(poi_df.limit(10))


StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 8, Finished, Available, Finished, False)

Ward filter (prod): none (empty ward list) — keeping all wards
RUN_SCOPE='partner portal': processing the business_service only
before join with poi_master, total: 204
poi_master coverage filter: kept 204/204 POIs present in both ai_poi and poi_master (dropped 0)
Backfill mode: (re-)enriching all scoped POIs (incl. newly-minted siblings)


SynapseWidget(Synapse.DataFrame, 4c20eabc-4379-4f6f-b9e5-28c1c08fa8cf)

#### Delete enrichment record of stale POI
 Instead of using a watermark system, this notebook based on record on poi_enrichment table. If there is an update in poi -> delete that record in poi_enrichment -> trigger the enrich again

In [7]:
%%sql
MERGE INTO poi_enrichment AS tgt
USING (
    SELECT DISTINCT p.poi_id
    FROM ai_poi p
    JOIN ai_enriched_poi ep
      ON ep.poi_id = p.poi_id
    WHERE p.updated_at > ep.enriched_at
      AND p.ingestion_source = 'partner_portal'
      AND ep.ingestion_source = 'partner_portal'
) AS src
ON tgt.poi_id = src.poi_id
WHEN MATCHED THEN DELETE

StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 9, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [8]:
# Incremental mode (moved here from the filter cell): drop any poi_id — source OR
# minted sibling — that already has a row in poi_enrichment. Sibling minting above
# always runs; only the LLM call is gated by incremental.
if RUN_MODE == 'incremental' and spark.catalog.tableExists(TARGET_POI_OUTPUT):
    _already_enriched = spark.table(TARGET_POI_OUTPUT).select('poi_id').distinct()
    _before = poi_df.count()
    poi_df = poi_df.join(_already_enriched, on='poi_id', how='left_anti')
    _after = poi_df.count()
    print(f'Incremental filter: skipped {_before - _after:,} already-enriched poi_id(s) (sources + siblings); {_after:,} left to enrich')
elif RUN_MODE == 'incremental':
    print(f'Incremental mode: {TARGET_POI_OUTPUT} does not exist yet — enriching all sources + siblings')
scoped_poi_ids = poi_df.select('poi_id').toPandas()['poi_id'].tolist()
print(f'Final working set after incremental filter: {len(scoped_poi_ids)} poi_id(s)')


StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 10, Finished, Available, Finished, False)

Final working set after incremental filter: 204 poi_id(s)


### Add Review Information

In [9]:
# Point from partner portal has no review

poi_pd = poi_df.toPandas()

if RUN_SCOPE != 'partner_portal':
    poi_review_pd = (
        poi_review_df
        .filter(F.col('poi_id').isin(scoped_poi_ids))
        .select(*review_cols)
        .toPandas()
    )
    if poi_review_pd.empty:
        review_agg_pd = pd.DataFrame(
            columns=['poi_id', 'review_texts', 'review_count', 'latest_review_date']
        )
    else:
        review_agg_pd = poi_review_pd.sort_values(
            ['review_date', 'updated_at'],
            ascending=[False, False],
            na_position='last',
        )
        review_agg_pd['review_compact'] = review_agg_pd.apply(compact_review_text, axis=1)
        review_agg_pd = review_agg_pd.groupby('poi_id', dropna=False).agg(
            review_texts=('review_compact', lambda v: [x for x in v.tolist() if x]),
            review_count=('review_id', 'count'),
            latest_review_date=('review_date', 'max'),
        ).reset_index()

    if poi_pd.empty:
        # No POIs survived the upstream filters — fall through with a clean empty frame so
        # downstream cells (cost rules, run_enrichment, persist, merge) can no-op safely.
        poi_pd = normalize_poi_dataframe(poi_pd)
        print({
            'poi_rows': 0,
            'review_rows': len(poi_review_pd),
            'dry_run_mode': DRY_RUN_MODE,
            'note': 'no POIs in working set — downstream stages will no-op',
        })
    else:
        poi_pd = poi_pd.merge(
            review_agg_pd,
            on='poi_id',
            how='left',
        )
        poi_pd = normalize_poi_dataframe(poi_pd)

        print({
            'poi_rows': len(poi_pd),
            'review_rows': len(poi_review_pd),
            'dry_run_mode': DRY_RUN_MODE,
        })
        display(poi_pd[['poi_id', 'name', 'primary_type', 'category_group', 'destination_name', 'rating', 'user_rating_count']].head(10))
else:
    # Partner-portal POIs have no reviews — skip the poi_review_df fetch/merge entirely,
    # but still create the same review columns the merge would have produced (empty/NaN),
    # so the schema matches the non-partner_portal path and normalize_poi_dataframe()
    # (which reads work['review_texts']) doesn't KeyError. Without this, downstream
    # cells (e.g. CostEstimator in cell 41) also KeyError on 'observed_cost_vnd',
    # 'primary_type', 'zone_name', 'category_group' — all created by normalize_poi_dataframe.
    poi_pd['review_texts'] = [[] for _ in range(len(poi_pd))]
    poi_pd['review_count'] = 0
    poi_pd['latest_review_date'] = pd.NaT

    poi_pd = normalize_poi_dataframe(poi_pd)

    print({
        'poi_rows': len(poi_pd),
        'review_rows': 0,
        'dry_run_mode': DRY_RUN_MODE,
        'note': 'partner_portal scope — reviews skipped, frame still normalized',
    })
    if not poi_pd.empty:
        display(poi_pd[['poi_id', 'name', 'primary_type', 'category_group', 'destination_name', 'rating', 'user_rating_count']].head(10))


StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 11, Finished, Available, Finished, False)

{'poi_rows': 204, 'review_rows': 0, 'dry_run_mode': False, 'note': 'partner_portal scope — reviews skipped, frame still normalized'}


SynapseWidget(Synapse.DataFrame, 6675e5fa-268e-4659-ae2a-c42939d44278)

## 4.2. Enrichment helpers

In [10]:
class CostEstimator:
    def __init__(self, df: pd.DataFrame):
        labeled = df[df['observed_cost_vnd'].notna()].copy()
        self.global_median = round_vnd(labeled['observed_cost_vnd'].median()) if not labeled.empty else None
        self.by_type = self._build_lookup(labeled, ['primary_type'])
        self.by_zone_type = self._build_lookup(labeled, ['zone_name', 'primary_type'])
        self.by_province_type = self._build_lookup(labeled, ['province_name', 'primary_type'])
        self.by_group = self._build_lookup(labeled, ['category_group'])
        self.by_price_level = self._build_lookup(labeled[labeled['price_level'].notna()], ['price_level'])
    
    @staticmethod
    def _build_lookup(df: pd.DataFrame, keys: list[str]) -> dict[Any, dict[str, Any]]:
        if df.empty:
            return {}
        
        grouped = df.groupby(keys)['observed_cost_vnd'].agg(['median', 'count']).reset_index()
        lookup = {}

        for _, row in grouped.iterrows():
            key = tuple(row[key] for key in keys) if len(keys) > 1 else row[keys[0]]
            lookup[key] = {'median': round_vnd(row['median']), 'count': int(row['count'])}

        return lookup
    
    def estimate(self, row: pd.Series) -> dict[str, Any]:
        observed = safe_int(row.get('observed_cost_vnd'))
        if observed is not None:
            return {'cost_vnd': observed, 'cost_method': 'observed_midpoint', 'confidence': 'high', 'evidence_count': 1}

        lookup_plan = [
            ((row.get('primary_type')), self.by_type, 5, 'type_median'),
            ((row.get('zone_name'), row.get('primary_type')), self.by_zone_type, 3, 'zone_type_median'),
            ((row.get('province_name'), row.get('primary_type')), self.by_province_type, 3, 'destination_type_median'),
            ((row.get('category_group')), self.by_group, 5, 'group_median'),
            ((row.get('price_level')), self.by_price_level, 3, 'price_level_median'),
        ]

        for key, lookup, min_count, method in lookup_plan:
            if key in lookup and lookup[key]['count'] >= min_count:
                confidence = 'high' if lookup[key]['count'] >= 8 else 'medium'
                return {
                    'cost_vnd': safe_int(lookup[key]['median']),
                    'cost_method': method,
                    'confidence': confidence,
                    'evidence_count': lookup[key]['count'],
                }

def format_best_time_label(best_time_code: str) -> str:
    return format_best_time_value(best_time_from_code(best_time_code))

def format_cost_text_from_value(row: pd.Series, cost_vnd):
    if cost_vnd is None or pd.isna(cost_vnd):
        low = safe_int(row.get('price_min'))
        high = safe_int(row.get('price_max'))
        if low is not None and high is not None:
            cost_vnd = round_vnd((low + high) / 2)
        else:
            return None
    
    cost_value = safe_int(cost_vnd)
    if cost_value is None:
        return None
    return f'{round_vnd(cost_value):,.0f}'

def format_cost_text(row: pd.Series):
    return format_cost_text_from_value(row, row.get('cost_vnd'))


def build_poi_payload(row):
    lang = row.get('lang') if row.get('lang') in SUPPORTED_LANGS else DEFAULT_LANG
    target_language = LANGUAGE_NAMES.get(lang, LANGUAGE_NAMES[DEFAULT_LANG])
    return {
        'poi_id': str(row['poi_id']),
        'lang': lang,
        'target_language': target_language,
        'name': row['name'],
        'primary_type': row['primary_type'],
        'types': row['types_norm'],
        'business_sector': row.get('business_sector'),
        'business_category': row.get('business_category'),
        'brand_or_chain': row.get('brand_or_chain'),
        'website_url': row.get('website_url'),
        'short_address': row.get('short_address'),
        'district': row.get('district'),
        'province_name': row.get('province_name'),
        'category_group': row['category_group'],
        'candidate_time_to_spend_minutes': int(round((row['duration_prior_min'] + row['duration_prior_max']) / 2)),
        'destination_name': row['destination_name'],
        'rating': None if pd.isna(row['rating']) else float(row['rating']),
        'user_rating_count': None if pd.isna(row['user_rating_count']) else int(row['user_rating_count']),
        'price_level': None if is_nullish(row.get('price_level')) else int(row.get('price_level')),
        'opening_hours_text': row.get('opening_hours_text', []),
        'service_flags': row.get('service_flags', []),
        'review_texts': row.get('review_texts', []),
        'candidate_best_time_code': row['best_time_candidate'],
        'candidate_best_time': best_time_from_code(row['best_time_candidate']),
        'deterministic_cost_vnd': None if is_nullish(row.get('cost_vnd')) else int(row.get('cost_vnd')),
        'deterministic_cost_method': row.get('cost_method'),
        'deterministic_cost_confidence': row.get('confidence'),
        'allowed_business_sectors': list(REF_TYPE_BY_SECTOR.keys()),
        'ref_type_by_sector': REF_TYPE_BY_SECTOR,
        'example_output_shape': POI_JSON_EXAMPLE,
        # Per-POI destination context — see DESTINATION_CONTEXTS in the configs cell.
        # Falls back to a generic Vietnam blurb when the POI's destination_id isn't mapped.
        'destination_context': DESTINATION_CONTEXTS.get(
            row.get('destination_id'),
            GENERIC_VIETNAM_CONTEXT,
        ),
    }


def normalize_model_result(row, payload, result):
    normalized = dict(result)
    normalized['poi_id'] = str(row['poi_id'])

    # Single-language free-text fields. The model is asked to write in the POI's own
    # language (payload['target_language']); we still accept legacy *_vi / *_en / *_ko keys as a
    # fallback so a stray bilingual response degrades gracefully into one value.
    lang = payload.get('lang', DEFAULT_LANG)
    normalized['description'] = clip_text(
        normalized.get('description') or normalized.get(f'description_{lang}') or '',
        DESCRIPTION_MAX_LEN,
    )

    normalized['time_to_spend_minutes'] = normalize_time_to_spend_minutes(
        normalized.get('time_to_spend_minutes'),
        payload['candidate_time_to_spend_minutes'],
    )
    best_time = normalize_best_time(
        normalized.get('best_time') or normalized.get('best_time_code'),
        payload['candidate_best_time_code'],
    )
    normalized['best_time'] = best_time
    normalized['best_time_type'] = best_time['type']
    normalized['best_time_from_month'] = best_time.get('from_month')
    normalized['best_time_to_month'] = best_time.get('to_month')
    normalized['best_time_value'] = format_best_time_value(best_time)

    normalized['experiences'] = normalize_experiences(
        normalized.get('experiences', normalized.get('experience')), row
    )

    deterministic_cost_vnd = payload['deterministic_cost_vnd']
    deterministic_confidence = payload['deterministic_cost_confidence']
    model_cost_vnd = safe_int(normalized.get('cost_vnd'))

    if deterministic_cost_vnd is not None and deterministic_confidence in {'high', 'medium'}:
        if model_cost_vnd is None:
            final_cost_vnd = deterministic_cost_vnd
            final_cost_method = payload['deterministic_cost_method']
        else:
            relative_gap = abs(model_cost_vnd - deterministic_cost_vnd) / max(deterministic_cost_vnd, 1)
            if relative_gap > 0.35:
                final_cost_vnd = deterministic_cost_vnd
                final_cost_method = payload['deterministic_cost_method']
            else:
                final_cost_vnd = round_vnd((model_cost_vnd + deterministic_cost_vnd) / 2)
                final_cost_method = f"{payload['deterministic_cost_method']}_llm_adjusted"
    elif model_cost_vnd is not None:
        final_cost_vnd = round_vnd(model_cost_vnd)
        final_cost_method = 'llm_inference'
    elif deterministic_cost_vnd is not None:
        final_cost_vnd = deterministic_cost_vnd
        final_cost_method = payload['deterministic_cost_method']
    else:
        final_cost_vnd = None
        final_cost_method = 'unresolved'

    normalized['review_summary'] = clip_text(
        normalized.get('review_summary') or normalized.get(f'review_summary_{lang}') or '',
        320,
    )
    normalized['confidence'] = normalize_allowed_string(
        normalized.get('confidence'), {'high', 'medium', 'low'}, 'medium', lowercase=True
    )
    normalized['reasoning_short'] = clip_text(normalized.get('reasoning_short', ''), 180)
    normalized['time_to_spend'] = format_time_to_spend(normalized['time_to_spend_minutes'])
    normalized['cost_vnd'] = final_cost_vnd
    normalized['cost_method'] = final_cost_method
    normalized['cost_confidence'] = deterministic_confidence if deterministic_confidence in {'high', 'medium'} and final_cost_method != 'llm_inference' else normalized['confidence']
    normalized['needs_manual_review'] = (
        normalized['confidence'] == 'low'
        or normalized['cost_confidence'] == 'low'
        or final_cost_vnd is None
        or final_cost_method in {'llm_inference', 'unresolved'}
    )
    ref_type_by_sector = payload.get('ref_type_by_sector') or {}
    raw_sector = normalized.get('ref_business_sector')
    raw_ref_type = normalized.get('ref_type')
    sector = raw_sector.strip() if isinstance(raw_sector, str) else None
    ref_type = raw_ref_type.strip() if isinstance(raw_ref_type, str) else None
    if sector in ref_type_by_sector:
        normalized['ref_business_sector'] = sector
        normalized['ref_type'] = ref_type if ref_type in ref_type_by_sector[sector] else None
    else:
         # Sector must never be null — default to 'other' when the LLM picks something unknown/missing.
        normalized['ref_business_sector'] = None
        normalized['ref_type'] = None
    return normalized


StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 12, Finished, Available, Finished, False)

## 4.3. Rule processing

In [11]:
if len(poi_pd)>0:
    cost_estimator = CostEstimator(poi_pd)

    cost_estimates = poi_pd.apply(cost_estimator.estimate, axis=1).apply(pd.Series)

    # poi_pd = pd.concat([poi_pd, cost_estimates], axis=1)
    for col in cost_estimates.columns:
        poi_pd[col] = cost_estimates[col].values

    poi_pd['cost'] = poi_pd.apply(format_cost_text, axis=1)
    poi_pd['best_time'] = poi_pd['best_time_candidate'].apply(format_best_time_label)

# Shared time-spend helpers used by prompt building and result normalization.
def normalize_time_to_spend_minutes(value, fallback_minutes, max_minutes: int = 1440) -> int:
    fallback_value = safe_int(fallback_minutes) or 90
    minutes = safe_int(value) or fallback_value
    return max(30, min(max_minutes, minutes))

def format_time_to_spend(minutes: int) -> str:
    minutes = normalize_time_to_spend_minutes(minutes, 90)
    return f'{minutes}'

print(f"lodging prior minutes: {DURATION_PRIORS['Lodging']}")

StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 13, Finished, Available, Finished, False)

lodging prior minutes: (1200, 1200)


## 4.4. LLM processing

In [12]:
POI_JSON_EXAMPLE = {
    "ref_business_sector": "f_n_b",
    "ref_type": "seafood_restaurant",
    "description": "One-sentence description in the target language, maximum 200 characters",
    "time_to_spend_minutes": 90,
    "best_time": {"type": "month_range", "from_month": 11, "to_month": 4},
    "cost_vnd": 250000,
    "experiences": [
        {"title": "Highlight 1", "description": "Concise and catchy highlight in the target language", "keyword": "travel_explore"},
        {"title": "Highlight 2", "description": "Concise and catchy highlight in the target language", "keyword": "shopping_bag"},
        {"title": "Highlight 3", "description": "Concise and catchy highlight in the target language", "keyword": "pending"},
    ],
    "review_summary": "Short summary of recurring review themes in the target language",
    "confidence": "medium",
    "reasoning_short": "One short sentence explaining confidence or uncertainty",
}

EXPERIENCE_KEYWORDS_SORTED = sorted(EXPERIENCE_KEYWORDS)

SYSTEM_PROMPT = f'''You are a senior travel content specialist for Vietnam.

Your job is to enrich Point of Interest records with concise, factual, traveler-friendly fields.
Use the supplied evidence first and do not invent unsupported details.
Produce every user-facing text field in ONE language only: the language given by `target_language` in the input payload (Vietnamese, English, or Korean). Do not output any other language, and do not produce parallel translations or `*_vi` / `*_en` / `*_ko` variants — emit a single value per text field.

Domain context: read the `destination_context` field from the input payload — it
describes the destination this POI belongs to (Da Nang, Hoi An, Lang Co, Ba Na Hills, Hue,
Phu Quoc, etc.) including seasonality, currency, major visitor zones, and notable landmarks.
Use those facts when picking best_time month ranges, cost_vnd, and descriptive prose.

Rules:
- Return only valid JSON.
- Return exactly these enrichment fields: ref_business_sector, ref_type, description, time_to_spend_minutes, best_time, cost_vnd, experiences, review_summary, confidence, reasoning_short.
- description, review_summary, and every experiences[].title / experiences[].description MUST be written in the target_language from the input payload.
- ref_business_sector must be exactly one value from allowed_business_sectors in the input payload. Pick the sector that best describes the POI's primary business, using poi_name, primary_type, types, business_sector, and business_category. Override the input business_sector if the evidence clearly points elsewhere.
- ref_business_sector must never be null. It must be exactly one value from allowed_business_sectors. Pick the sector that best describes the POI's primary business, using poi_name, primary_type, types, business_sector, and business_category. Override the input business_sector if the evidence clearly points elsewhere. If no specific sector fits, return 'other'.
- ref_type must be exactly one value from ref_type_by_sector[ref_business_sector]. If a sector fits but no ref_type within it clearly fits, return ref_type as null (ref_business_sector still set). If no ref_business_sector clearly fits the POI, return both ref_business_sector and ref_type as null.
- Do not invent values outside the allowed lists.
- description must be one sentence and at most 200 characters.
- The description should briefly describe the place or its best attraction and catch the user's attention.
- The description omits the place name.
- time_to_spend_minutes must be a realistic duration estimate for the POI experience.
- For lodging records, time_to_spend_minutes should represent the estimated total stay between check-in and check-out.
- Lodging time_to_spend_minutes may therefore be much larger than attraction or dining durations.
- best_time must be an object: {{'type': 'all_year', 'from_month': null, 'to_month': null}} or {{'type': 'month_range', 'from_month': 1-12, 'to_month': 1-12}}.
- candidate_best_time is only a fallback hint. Do not blindly copy it; refine or override the month range based on the POI's actual characteristics and evidence.
- Use all_year only when seasonality is not meaningful for the POI, such as many indoor, shopping, dining, or spa records.
- Use a specific month_range for seasonal POIs. Examples: sea/island/boat activities may favor calmer dry months, snorkeling/diving may use a narrower visibility window, waterfalls may favor greener/rainier months, and hiking may avoid the wettest months.
- Cross-year ranges are allowed, for example November to April as from_month=11 and to_month=4.
- cost_vnd must be an integer in VND for a typical adult visit.
- For Sunworld Bana Hill, cost will alway equal 1000000
- experiences must contain exactly 3 POI highlights based on cultural or touristic characteristics.
- Each experience must include title, description, and keyword, with title and description in the target_language.
- Each title must be at most 50 characters and each description at most 150 characters.
- Each experience keyword MUST be exactly one value from this list (Material Design icon names; pick the one whose icon best represents the highlight): {EXPERIENCE_KEYWORDS_SORTED}.
- Do not invent, translate, pluralize, or modify the keyword in any way. If no value in the list fits, return "pending".
- review_summary must reflect the overall theme of the comments, not isolated remarks. Leave it null if no reviews are available.
- If deterministic_cost_confidence is high or medium, keep cost_vnd close to deterministic_cost_vnd unless there is strong place-specific evidence to adjust it.
- If deterministic_cost_confidence is low or deterministic_cost_vnd is null, you may infer cost from the POI name, brand, category, venue type, price level, and broader knowledge.
- If evidence is weak, lower confidence and keep wording conservative.
'''

print('Prompt templates ready')


import random


def _parse_retry_after(value):
    """Azure sends Retry-After as integer seconds. Returns None if absent/non-numeric."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


class RateLimitError(Exception):
    def __init__(self, message, retry_after=None):
        super().__init__(message)
        self.retry_after = retry_after


def _retry_wait(retry_state):
    exc = retry_state.outcome.exception()
    ra = getattr(exc, 'retry_after', None)
    base = ra if ra is not None else min(2 * (2 ** (retry_state.attempt_number - 1)), 30)
    return base + random.uniform(0, min(base, 5))


async def call_llm_async(session: aiohttp.ClientSession, system_prompt: str, user_payload: dict[str, Any], poi_id: str) -> dict[str, Any]:
    url = f'{AZURE_AI_ENDPOINT}/openai/deployments/{AZURE_AI_DEPLOYMENT}/chat/completions?api-version={AZURE_AI_API_VERSION}'
    headers = {'Content-Type': 'application/json', 'api-key': AZURE_AI_API_KEY}
    payload = {
        'model': AZURE_AI_DEPLOYMENT,
        'max_tokens': MAX_TOKENS,
        'temperature': 0.2,
        'response_format': {'type': 'json_object'},
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': json.dumps(user_payload, ensure_ascii=False)},
        ],
    }
    async with session.post(url, headers=headers, json=payload) as resp:
        if resp.status == 429:
            raise RateLimitError(
                f'Rate limited on poi_id={poi_id}',
                retry_after=_parse_retry_after(resp.headers.get('Retry-After')),
            )
        if resp.status >= 400:
            body = await resp.text()
            raise ValueError(f'HTTP {resp.status}: {body[:300]}')
        data = await resp.json()
        raw_text = data['choices'][0]['message']['content']
        raw_text = re.sub(r'^```json\s*|\s*```$', '', raw_text.strip())
        return json.loads(raw_text)


async def enrich_single_poi(session: aiohttp.ClientSession, semaphore: asyncio.Semaphore, row: pd.Series) -> dict[str, Any]:
    poi_id = str(row.get('poi_id', 'unknown'))
    user_payload = build_poi_payload(row)
    async with semaphore:
        try:
            model_result = None
            async for attempt in AsyncRetrying(
                stop=stop_after_attempt(MAX_RETRIES),
                wait=_retry_wait,
                retry=retry_if_exception_type(
                    (RateLimitError, aiohttp.ClientError, asyncio.TimeoutError)
                ),
            ):
                with attempt:
                    model_result = await call_llm_async(session, SYSTEM_PROMPT, user_payload, poi_id)
            final_record = normalize_model_result(row, user_payload, model_result)
            final_record['enriched_at'] = datetime.now(timezone.utc).isoformat()
            final_record['status'] = 'success'
            return final_record
        except Exception as exc:
            return {
                'poi_id': str(row.get('poi_id', 'unknown')),
                'poi_name': row.get('poi_name', ''),
                'description': '',
                'review_summary': '',
                'experiences': [],
                'status': 'failed',
                'needs_manual_review': True,
                'error_message': str(exc)[:500],
                'enriched_at': datetime.now(timezone.utc).isoformat(),
            }


async def enrich_batch(batch_df: pd.DataFrame, semaphore: asyncio.Semaphore) -> list[dict[str, Any]]:
    connector = aiohttp.TCPConnector(limit=CONCURRENCY_LIMIT)
    timeout = aiohttp.ClientTimeout(total=90)
    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        tasks = [enrich_single_poi(session, semaphore, row) for _, row in batch_df.iterrows()]
        return list(await asyncio.gather(*tasks))

async def run_enrichment(df: pd.DataFrame) -> pd.DataFrame:
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
    batches = [df.iloc[i:i + BATCH_SIZE] for i in range(0, len(df), BATCH_SIZE)]
    all_results: list[dict[str, Any]] = []

    for batch_idx, batch_df in enumerate(batches, start=1):
        started = time.time()
        batch_results = await enrich_batch(batch_df, semaphore)
        all_results.extend(batch_results)
        elapsed = round(time.time() - started, 1)
        ok_count = sum(1 for r in batch_results if r['status'] == 'success')
        fallback_count = len(batch_results) - ok_count
        print(f'Batch {batch_idx}/{len(batches)}: {ok_count} success, {fallback_count} fallback, {elapsed}s')

    return pd.DataFrame(all_results)

poi_enriched_pd = await run_enrichment(poi_pd)
print(f'Enriched {len(poi_enriched_pd):,} POIs')
display(poi_enriched_pd.head(20))


StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 14, Finished, Available, Finished, False)

Prompt templates ready
Batch 1/6: 40 success, 0 fallback, 14.9s
Batch 2/6: 40 success, 0 fallback, 15.0s
Batch 3/6: 40 success, 0 fallback, 81.3s
Batch 4/6: 40 success, 0 fallback, 145.4s
Batch 5/6: 40 success, 0 fallback, 15.7s
Batch 6/6: 4 success, 0 fallback, 3.3s
Enriched 204 POIs


SynapseWidget(Synapse.DataFrame, 31a7874a-e586-479e-8342-8dbc3f53c03a)

## 4.5. Gold patch

In [13]:
current_ts = pd.Timestamp.now(tz='UTC')

if len(poi_pd)>0:
    poi_final_pd = poi_pd.merge(poi_enriched_pd, on='poi_id', how='inner', suffixes=('_source', ''))


    ENRICHMENT_JOB_ID = f"poi_enrichment_{current_ts.strftime('%Y%m%dT%H%M%SZ')}"

    source_updated_col = 'updated_at_source' if 'updated_at_source' in poi_final_pd.columns else 'updated_at'
    source_updated_at = poi_final_pd[source_updated_col] if source_updated_col in poi_final_pd.columns else pd.NaT
    status_map = {'success': 'success', 'fallback': 'failed'}


    # Persist enriched fields as raw values + metadata, one row per POI in the POI's own language.
    # The content language comes from the source `lang` column (vi, en, or ko); there is no language fan-out —
    # each POI is enriched once, in its own language, and emitted as a single row.
    # No display formatting here — the poi_enrichment_display table handles that downstream.


    def _serialize_experiences(value):
        if is_nullish(value):
            return None
        return json.dumps(value, ensure_ascii=False)


    def _safe_month_int(value):
        return None if pd.isna(value) else int(value)

    def _truncate(value, max_len=500, suffix='...'):
        if is_nullish(value):
            return None
        value = str(value)
        if len(value) <= max_len:
            return value
        cut = value[:max_len - len(suffix)]
        # back off to the last full word if there's a space to break on
        if ' ' in cut:
            cut = cut.rsplit(' ', 1)[0]
        return cut + suffix


    _enrichment_records = []
    for _, src in poi_final_pd.iterrows():
        poi_id = str(src['poi_id'])
        lang = src.get('lang') if src.get('lang') in SUPPORTED_LANGS else DEFAULT_LANG
        ingestion_source = src.get('ingestion_source')
        enrichment_status = status_map.get(src.get('status'), 'failed')
        validation_status = 'warning' if bool(src.get('needs_manual_review')) else 'valid'
        row_source_updated_at = src.get(source_updated_col) if source_updated_col in poi_final_pd.columns else pd.NaT
        description_value = _truncate(src.get('description'))


        _enrichment_records.append({
            'enrichment_id': f'{ENRICHMENT_JOB_ID}_{poi_id}_{lang}',
            'poi_id': poi_id,
            'poi_name': src.get('poi_name'),
            'lang': lang,
            'enrichment_job_id': ENRICHMENT_JOB_ID,
            'enrichment_status': enrichment_status,
            'validation_status': validation_status,
            'model_name': AZURE_AI_DEPLOYMENT,
            'source_updated_at': row_source_updated_at,
            'ref_business_sector': src.get('ref_business_sector'),
            'ref_type': src.get('ref_type'),
            'description': description_value,
            'time_to_spend_minutes': src.get('time_to_spend_minutes'),
            'best_time_type': src.get('best_time_type'),
            'best_time_from_month': _safe_month_int(src.get('best_time_from_month')),
            'best_time_to_month': _safe_month_int(src.get('best_time_to_month')),
            'cost_vnd': src.get('cost_vnd'),
            'cost_method': src.get('cost_method'),
            'cost_confidence': src.get('cost_confidence'),
            'experiences': _serialize_experiences(src.get('experiences')),
            'review_summary': src.get('review_summary'),
            'reasoning_short': src.get('reasoning_short'),
            'enriched_at': current_ts,
            'is_current': True,
        })

    poi_enrichment_pd = pd.DataFrame(_enrichment_records)


    # Compare original type vs ref_type (LLM-assigned) for QA. Uses poi_final_pd which has both
    # the source POI columns and the enriched columns merged together.
    _ref_type_check = poi_final_pd[['poi_id', 'poi_name', 'business_sector', 'ref_business_sector', 'business_category', 'poi_types', 'ref_type']].rename(
        columns={'poi_name': 'name', 'business_category': 'type'}
    )

    _total = len(_ref_type_check)
    _with_ref = _ref_type_check['ref_type'].notna().sum()
    print('preview comparision')
    display(_ref_type_check)
    print(f'ref_type populated: {_with_ref:,} / {_total:,}  ({_with_ref / _total:.1%} coverage)')
    print()
    print('Per-ref_business_sector coverage (not-null ref_type rate within each LLM-picked sector):')
    print(
        _ref_type_check.groupby('ref_business_sector', dropna=False)
        .agg(total=('poi_id', 'count'), with_ref=('ref_type', lambda s: s.notna().sum()))
        .assign(coverage=lambda d: (d['with_ref'] / d['total']).map('{:.1%}'.format))
        .sort_values('total', ascending=False)
        .to_string()
    )
    print()
    print('Sector reclassification — where ref_business_sector differs from the source business_sector:')
    _mismatched = _ref_type_check[_ref_type_check['business_sector'].fillna('') != _ref_type_check['ref_business_sector'].fillna('')]
    print(f'  {len(_mismatched):,} / {_total:,} POIs reclassified ({len(_mismatched) / _total:.1%})')
    if len(_mismatched):
        display(_mismatched[['poi_id', 'name', 'business_sector', 'ref_business_sector', 'type', 'ref_type']].head(20))
    print()
    print('Sample (one row per ref_type, randomly):')
    display(
        _ref_type_check.dropna(subset=['ref_type'])
            .groupby('ref_type').first().reset_index()
            [['poi_id', 'name', 'business_sector', 'ref_business_sector', 'type', 'poi_types', 'ref_type']]
    )
    print()
    print('POIs where ref_type is null:')
    display(_ref_type_check[_ref_type_check['ref_type'].isna()].head(20))



    from pyspark.sql.types import (
        StructType, StructField,
        StringType, LongType, DoubleType, BooleanType, TimestampType,
    )

    # Explicit schema for every enriched field. Types match the existing
    # production Delta table exactly so appends won't trigger
    # DELTA_FAILED_TO_MERGE_FIELDS. Update this in lockstep with any prod
    # schema migration.
    POI_ENRICHMENT_SCHEMA = StructType([
        StructField('enrichment_id',         StringType(),    False),
        StructField('poi_id',                StringType(),    False),
        StructField('poi_name',              StringType(),    True),
        StructField('lang',                  StringType(),    True),
        StructField('enrichment_job_id',     StringType(),    False),
        StructField('enrichment_status',     StringType(),    False),
        StructField('validation_status',     StringType(),    False),
        StructField('model_name',            StringType(),    True),
        StructField('source_updated_at',     TimestampType(), True),
        StructField('ref_business_sector',   StringType(),    True),
        StructField('ref_type',              StringType(),    True),
        StructField('description',           StringType(),    True),
        StructField('time_to_spend_minutes', LongType(),      True),
        StructField('best_time_type',        StringType(),    True),
        StructField('best_time_from_month',  DoubleType(),    True),
        StructField('best_time_to_month',    DoubleType(),    True),
        StructField('cost_vnd',              LongType(),      True),
        StructField('cost_method',           StringType(),    True),
        StructField('cost_confidence',       StringType(),    True),
        StructField('experiences',           StringType(),    True),
        StructField('review_summary',        StringType(),    True),
        StructField('reasoning_short',       StringType(),    True),
        StructField('enriched_at',           TimestampType(), False),
        StructField('is_current',            BooleanType(),   False),
    ])

    if poi_enrichment_pd.empty:
        # Empty path: skip Spark conversion entirely (spark.createDataFrame can't infer a
        # schema from an empty pandas frame). Build an empty Spark DataFrame matching the
        # target schema so downstream cells can still reference poi_enrichment_df.
        poi_enrichment_df = spark.createDataFrame([], schema=POI_ENRICHMENT_SCHEMA)
    else:
        # Align column order/types to the explicit schema (forward-compatible — extra
        # pandas columns are dropped, missing columns are filled with NULL via the cast).
        _pd_cols = set(poi_enrichment_pd.columns)
        for _f in POI_ENRICHMENT_SCHEMA.fields:
            if _f.name not in _pd_cols:
                poi_enrichment_pd[_f.name] = None
        poi_enrichment_df = spark.createDataFrame(
            poi_enrichment_pd[[f.name for f in POI_ENRICHMENT_SCHEMA.fields]]
        )
        poi_enrichment_df = poi_enrichment_df.select(*[
            F.col(field.name).cast(field.dataType).alias(field.name)
            for field in POI_ENRICHMENT_SCHEMA.fields
        ])

    poi_enrichment_df.select('poi_id').distinct().createOrReplaceTempView('current_enriched_poi_ids')

    if not DRY_RUN_MODE and spark.catalog.tableExists(TARGET_POI_OUTPUT):
        spark.sql(f"""
            MERGE INTO {TARGET_POI_OUTPUT} AS target
            USING current_enriched_poi_ids AS source
            ON target.poi_id = source.poi_id AND target.is_current = TRUE
            WHEN MATCHED THEN UPDATE SET target.is_current = FALSE
        """)

    write_mode = 'overwrite' if DRY_RUN_MODE else 'append'
    poi_enrichment_df.write.mode(write_mode).option('mergeSchema', 'true').saveAsTable(TARGET_POI_OUTPUT)


StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 15, Finished, Available, Finished, False)

preview comparision


SynapseWidget(Synapse.DataFrame, 22fe595f-11e9-4b2e-a881-3af49cd4b72e)

ref_type populated: 194 / 204  (95.1% coverage)

Per-ref_business_sector coverage (not-null ref_type rate within each LLM-picked sector):
                     total  with_ref coverage
ref_business_sector                          
accommodation          162       158    97.5%
f_n_b                   27        23    85.2%
attraction               5         5   100.0%
vehicle_rental           4         4   100.0%
other                    2         0     0.0%
shopping_n_retail        2         2   100.0%
event                    1         1   100.0%
spa_n_wellness           1         1   100.0%

Sector reclassification — where ref_business_sector differs from the source business_sector:
  7 / 204 POIs reclassified (3.4%)


SynapseWidget(Synapse.DataFrame, d9ae5dfc-d93a-4be0-a14a-4d7d34e98cbd)


Sample (one row per ref_type, randomly):


SynapseWidget(Synapse.DataFrame, 008c5136-a7ba-47da-8b77-4db55c386239)


POIs where ref_type is null:


SynapseWidget(Synapse.DataFrame, 84e68799-1ed9-45d5-938e-cb618ad545d1)

##### Fix mismatch duing renrichment due to poi en generation happen before enrichment process (enrichment process running parallely for vi and en)

In [14]:
%%sql
MERGE INTO poi_enrichment AS target
USING (
    WITH pp AS (
        SELECT poi_id, source_id, lang
        FROM (
            SELECT
                poi_id, source_id, lang,
                ROW_NUMBER() OVER (
                    PARTITION BY source_id, lang
                    ORDER BY updated_at DESC
                ) AS rn
            FROM ai_poi
            WHERE lower(trim(ingestion_source)) = 'partner_portal'
              AND source_id IS NOT NULL
        )
        WHERE rn = 1
    ),
    joined AS (
        SELECT
            en.poi_id AS poi_id_en,
            pe_vi.ref_business_sector,
            pe_vi.ref_type,
            pe_vi.time_to_spend_minutes,
            pe_vi.best_time_type,
            pe_vi.best_time_from_month,
            pe_vi.best_time_to_month,
            pe_vi.cost_vnd,
            ROW_NUMBER() OVER (PARTITION BY en.poi_id ORDER BY vi.poi_id) AS _rn
        FROM pp vi
        JOIN poi_enrichment pe_vi
          ON pe_vi.poi_id = vi.poi_id AND pe_vi.is_current = true AND vi.lang = 'vi'
        JOIN pp en
          ON en.source_id = vi.source_id AND en.lang = 'en'
        JOIN poi_enrichment pe_en
          ON pe_en.poi_id = en.poi_id AND pe_en.is_current = true
        WHERE pe_vi.ref_business_sector   IS DISTINCT FROM pe_en.ref_business_sector
           OR pe_vi.ref_type              IS DISTINCT FROM pe_en.ref_type
           OR pe_vi.time_to_spend_minutes IS DISTINCT FROM pe_en.time_to_spend_minutes
           OR pe_vi.best_time_type        IS DISTINCT FROM pe_en.best_time_type
           OR pe_vi.best_time_from_month  IS DISTINCT FROM pe_en.best_time_from_month
           OR pe_vi.best_time_to_month    IS DISTINCT FROM pe_en.best_time_to_month
           OR pe_vi.cost_vnd              IS DISTINCT FROM pe_en.cost_vnd
    )
    SELECT poi_id_en, ref_business_sector, ref_type, time_to_spend_minutes,
           best_time_type, best_time_from_month, best_time_to_month, cost_vnd
    FROM joined
    WHERE _rn = 1
) AS source
ON target.poi_id = source.poi_id_en AND target.is_current = true
WHEN MATCHED THEN UPDATE SET
    ref_business_sector    = source.ref_business_sector,
    ref_type                = source.ref_type,
    time_to_spend_minutes  = source.time_to_spend_minutes,
    best_time_type         = source.best_time_type,
    best_time_from_month   = source.best_time_from_month,
    best_time_to_month     = source.best_time_to_month,
    cost_vnd                = source.cost_vnd

StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 16, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

##### Check again 

In [15]:
%%sql
WITH pp AS (
    SELECT poi_id, source_id, lang
    FROM ai_poi
    WHERE lower(trim(ingestion_source)) = 'partner_portal'
      AND source_id IS NOT NULL
)
SELECT
    vi.source_id,
    pe_vi.poi_id AS poi_id_vi,
    pe_en.poi_id AS poi_id_en,
    pe_vi.ref_business_sector        AS vi_business_sector,
    pe_en.ref_business_sector        AS en_business_sector,
    pe_vi.ref_type                   AS vi_business_category,
    pe_en.ref_type                   AS en_business_category,
    pe_vi.time_to_spend_minutes      AS vi_time_to_spend_minutes,
    pe_en.time_to_spend_minutes      AS en_time_to_spend_minutes,
    pe_vi.best_time_type             AS vi_best_time_type,
    pe_en.best_time_type             AS en_best_time_type,
    pe_vi.best_time_from_month       AS vi_best_time_from_month,
    pe_en.best_time_from_month       AS en_best_time_from_month,
    pe_vi.best_time_to_month         AS vi_best_time_to_month,
    pe_en.best_time_to_month         AS en_best_time_to_month,
    pe_vi.cost_vnd                   AS vi_cost_vnd,
    pe_en.cost_vnd                   AS en_cost_vnd
FROM pp vi
JOIN poi_enrichment pe_vi
  ON pe_vi.poi_id = vi.poi_id AND pe_vi.is_current = true AND vi.lang = 'vi'
JOIN pp en
  ON en.source_id = vi.source_id AND en.lang = 'en'
JOIN poi_enrichment pe_en
  ON pe_en.poi_id = en.poi_id AND pe_en.is_current = true
WHERE pe_vi.ref_business_sector   IS DISTINCT FROM pe_en.ref_business_sector
   OR pe_vi.ref_type              IS DISTINCT FROM pe_en.ref_type
   OR pe_vi.time_to_spend_minutes IS DISTINCT FROM pe_en.time_to_spend_minutes
   OR pe_vi.best_time_type        IS DISTINCT FROM pe_en.best_time_type
   OR pe_vi.best_time_from_month  IS DISTINCT FROM pe_en.best_time_from_month
   OR pe_vi.best_time_to_month    IS DISTINCT FROM pe_en.best_time_to_month
   OR pe_vi.cost_vnd              IS DISTINCT FROM pe_en.cost_vnd

StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 17, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 17 fields>

### 4.7 ai_enriched_poi merge 

In [16]:
if len(poi_pd)>0:
    # --- 4.7 ai_enriched_poi merge (§3.7) ---
    # Build the 103-column ai_enriched_poi row per (poi_id, lang) for everything we just
    # enriched. Maps poi_enrichment.* → ai_enriched_poi.ref_* via an explicit rename map.
    # Upsert by poi_id, like the display table.

    # Self-contained imports so this cell can run standalone after a kernel restart
    # (cell 0 / configs cells may not have been executed).
    import pyspark.sql.functions as F

    # ai_enriched_poi write target — dry-run writes to ai_enriched_poi_dryrun (see the Fabric
    # target-tables config). Standalone runs default to production unless DRY_RUN_MODE is set.
    globals().setdefault('TARGET_AI_ENRICHED_POI_OUTPUT',
                        'ai_enriched_poi_dryrun' if DRY_RUN_MODE else AI_ENRICHED_POI_TABLE)
    # TARGET_POI_OUTPUT is the enrichment write target (toggled by DRY_RUN_MODE). When
    # running this cell standalone we always read from the production enrichment table.
    globals().setdefault('TARGET_POI_OUTPUT',     POI_ENRICHMENT_TABLE)
    # RUN_SCOPE-safe default for standalone reruns (mirrors the TARGET_* setdefault pattern
    # above). 'partner_portal' further restricts the merge below to partner-portal poi_ids only.
    globals().setdefault('RUN_SCOPE', 'full')

    # --- Label → enum lookup tables (per "Mapping data trên KB" Confluence doc) ---
    # The LLM's ref_type / ref_business_sector output can come back as the human-readable
    # poi_master English label (e.g. "Traditional Taxi") rather than the canonical
    # snake_case enum (e.g. "taxi_service"). These maps normalize label -> enum before the
    # value is written into ai_enriched_poi.business_category / business_sector.
    CATEGORY_LABEL_TO_ENUM = {
        # Accommodation
        "Hotel": "hotel", "Extended Stay Hotel": "extended_stay_hotel", "Inn": "inn",
        "Resort": "resort_hotel", "Hostel": "hostel", "Motel": "motel",
        "Homestay": "guest_house", "Farmstay": "farmstay", "Campground/Glamping": "campground", "Campground/ Glamping": "campground",  # schema spacing variant
        "Camping Cabin": "camping_cabin", "Bed & Breakfast": "bed_and_breakfast",
        "Cottage": "cottage", "Mobile Home Park": "mobile_home_park",
        # F&B
        "Vietnamese Restaurant": "vietnamese_restaurant", "Chinese Restaurant": "chinese_restaurant",
        "Japanese Restaurant": "japanese_restaurant", "Korean Restaurant": "korean_restaurant",
        "Thai Restaurant": "thai_restaurant", "French Restaurant": "french_restaurant",
        "Italian Restaurant": "italian_restaurant", "American Restaurant": "american_restaurant",
        "Western Restaurant": "western_restaurant", "International Restaurant": "internation_restaurant",
        # Partner Portal's actual main_cuisine enum has NO "Restaurant"/"General" suffix
        # (schema: f_n_b.json /facilities/main_cuisine) -- these are the real raw values.
        "Vietnamese": "vietnamese_restaurant", "Chinese": "chinese_restaurant",
        "Japanese": "japanese_restaurant", "Korean": "korean_restaurant",
        "Thai": "thai_restaurant", "French": "french_restaurant",
        "Italian": "italian_restaurant", "American": "american_restaurant",
        "Western General": "western_restaurant", "International": "internation_restaurant",
        "Fusion": "fusion_restaurant", "Bakery & Dessert": "bakery_dessert",
        "Bar & Pub/Nightlife": "night_club", "Bar & Pub / Nightlife": "night_club",
        "Bar & Pub/ Nightlife": "night_club",  # schema spacing variant (f_n_b.json main_cuisine)
        "Fast Food": "fast_food",
        "Vegan Restaurant": "vegetarian_restaurant", "Seafood Restaurant": "seafood_restaurant",
        "Café": "cafe", "Hot Pot": "hot_pot_restaurant", "Barbecue & Grill": "barbecue_restaurant",
        "Noodle": "noodle_shop", "Fine Dining": "fine_dining_restaurant", "Buffet": "buffet_restaurant",
        "Brunch & Breakfast": "brunch_restaurant", "Halal": "halal_restaurant",
        "Gastropub & Brewery": "gastropub",
        # Spa & Wellness
        "Day Spa": "spa",  # schema raw value (spa_n_wellness.json business_category)
        "Therapeutic Massage": "therapeutic_massage",
        "Clinic-Aesthetic Spa": "clinic_aesthetic_spa", "Blind Massage": "blind_massage",
        "Onsen-Jjimjilbang": "onsen", "Hair Salon": "hair_salon", "Nail-Lash Salon": "nail_salon",
        "Gym-Fitness": "fitness_center", "Yoga-Pilates Studio": "yoga_pilates_studio",
        "Meditation Center": "meditation_center", "Beauty Salon": "beauty_salon",
        "Hair Removal Service": "hair_removal_service", "Tanning Studio": "tanning_studio",
        "Barber Shop": "barber_shop", "Barbershop": "barber_shop",  # doc EN display is one word
        "Tattoo & Piercing": "body_piercing_shop",
        "Swimming Pool": "swimming_pool",
        # Shopping & Retail
        "Local Specialty": "local_specialty", "Souvenirs & Handicrafts": "souvenir_handicraft",
        "Fashion & Accessories": "fashion_accessories", "Jewelry & Gems": "jewelry_store",
        "Cosmetics & Beauty": "cosmetics_store", "Books & Stationery": "book_store",
        "Mall - Supermarket": "shopping_mall", "Mall-Supermarket": "shopping_mall",  # schema variant
        "Convenience Store": "convenience_store",
        # New in schema (shopping_n_retail.json main_category), not yet in the Confluence doc
        "Packaged Food": "packaged_food", "Home & Decor": "home_decor", "Electronics": "electronics_store",
        "Local/Night Market": "local_market", "Local/ Night Market": "local_market",  # doc spacing variant
        "Sporting Goods": "sporting_goods_store",
        "Toy Store": "toy_store", "Luggage Store": "luggage_store", "Florist": "florist",
        # Attraction
        "Theme Park": "theme_park", "Water Park": "water_park", "Museum": "museum",
        "Historical Site": "historical_place", "Zoo-Aquarium": "zoo_and_aquarium",
        "Nature Reserve": "national_park", "Architecture-Landmark": "landmark",
        "Art Show-Performance": "art_show_performance", "Traditional Village": "traditional_village",
        "Religious Site": "religious_site", "Casino": "casino", "Art Gallery": "art_gallery",
        "Eco-tourism": "eco_tourism",  # new in schema (attraction.json main_category)
        "Cultural Center": "cultural_center", "Indoor Entertainment": "indoor_entertainment",
        # Vehicle Rental
        "Motorbike Rental": "motocycle_rental_agency", "Self-drive Car Rental": "self_drive_car_rental",
        "Car with Driver": "chauffeur_car_rental", "Bicycle-EV Rental": "bicycle_ev_rental",
        "Campervan Rental": "campervan_rental", "Boat Rental": "boat_rental",
        # Event
        "Music & Entertainment": "music_entertainment_event", "Sports": "sports_event",
        "Education & Business": "education_business_event",
        "Festival & Community": "festival_community_event", "Arts & Culture": "arts_culture_event",
        # Transport Route-base
        "Sleeper Bus": "sleeper_bus_service", "Limousine Van": "limousine_service",
        "Speedboat": "speedboat_service", "Ferry": "ferry_service", "Train": "train_service",
        "Flight": "flight",
        # Transport On-demand
        "Traditional Taxi": "taxi_service", "Chauffeur & Charter Company": "chauffeur_charter_service",
        "Tech-hailing Platform": "ride_hailing_platform",
        "Airport Limousine Service": "airport_limousine_service",
        # Fallback (doc §1.2, steps 3-4): business_category = "Other" when nothing else maps
        "Other": "other",
    }
    BUSINESS_SECTOR_LABEL_TO_ENUM = {
        "F&B": "f_n_b", "Shopping & Retail": "shopping_n_retail", "Accommodation": "accommodation",
        "Accomodation": "accommodation",  # schema typo (accomodation.json category.en) -- real raw value
        "Tour & Activities": "tour_n_activities", "Attraction": "attraction",
        "Spa & Wellness": "spa_n_wellness", "Spa": "spa_n_wellness",  # schema uses "Spa" (spa_n_wellness.json category.en)
        "Transport On Demand": "transport_on_demand", "Transport On-demand": "transport_on_demand",  # schema variant
        "Transport Route Base": "transport_route_base", "Transport Route-base": "transport_route_base",  # schema variant
        "Vehicle Rental": "vehicle_rental",
        "Event": "event",
        # Fallback sector (doc §1.1 mapping table, "Other" row)
        "Other": "other",
    }
    _cat_map_expr = F.create_map(*[x for kv in CATEGORY_LABEL_TO_ENUM.items() for x in (F.lit(kv[0]), F.lit(kv[1]))])
    _sector_map_expr = F.create_map(*[x for kv in BUSINESS_SECTOR_LABEL_TO_ENUM.items() for x in (F.lit(kv[0]), F.lit(kv[1]))])

    # Drive the merge off the poi_enrichment TABLE (single source of truth), not off the
    # in-memory poi_enrichment_pd. This way the merge is idempotent and lets ai_enriched_poi
    # catch up even on runs that didn't produce new enrichments. Mirrors merge_script.sql.
    if not spark.catalog.tableExists(TARGET_POI_OUTPUT):
        print(f'Skipping ai_enriched_poi merge — {TARGET_POI_OUTPUT} does not exist yet.')
    else:
        enrichment_scope_df = (
            spark.table(TARGET_POI_OUTPUT)
                .filter(F.col('is_current') == True)
        )
        _enriched_poi_ids = (
            enrichment_scope_df.select('poi_id').distinct()
                            .toPandas()['poi_id'].astype(str).tolist()
        )

        # RUN_SCOPE='partner_portal': never full-merge — restrict to partner-portal poi_ids
        # only, so this run doesn't touch/upsert rows belonging to other ingestion sources
        # even though poi_enrichment.is_current may include them from earlier full/test runs.
        if RUN_SCOPE == 'partner_portal':
            _partner_portal_poi_ids = (
                spark.table(POI_TABLE)
                    .filter(F.lower(F.trim(F.col('ingestion_source'))) == PARTNER_PORTAL_SOURCE)
                    .select('poi_id').distinct()
                    .toPandas()['poi_id'].astype(str).tolist()
            )
            _enriched_poi_ids = sorted(set(_enriched_poi_ids) & set(_partner_portal_poi_ids))
            print(f"RUN_SCOPE='partner_portal': restricted ai_enriched_poi merge to "
                f"{len(_enriched_poi_ids)} partner-portal poi_id(s) (no full merge)")

        # Restrict to poi_id(s) that have a non-null destination_id — ai_enriched_poi rows
        # without a destination can't be displayed/routed downstream, so skip minting/updating
        # them here. Applied after the partner_portal restriction so it narrows whatever scope
        # was already selected above.
        _destination_poi_ids = (
            spark.table(POI_TABLE)
                .filter(F.col('destination_id').isNotNull())
                .select('poi_id').distinct()
                .toPandas()['poi_id'].astype(str).tolist()
        )
        _n_before_dest_filter = len(_enriched_poi_ids)
        _enriched_poi_ids = sorted(set(_enriched_poi_ids) & set(_destination_poi_ids))
        print(f'destination_id filter: {_n_before_dest_filter} -> {len(_enriched_poi_ids)} '
              f'poi_id(s) after excluding NULL destination_id')


        if not _enriched_poi_ids:
            print('Skipping ai_enriched_poi merge — poi_enrichment has no is_current rows.')
            # Use a dummy ai_poi_scope_df so the rest of the cell can no-op cleanly.
            ai_poi_scope_df = spark.table(POI_TABLE).filter(F.lit(False))
        else:
            # Keep vi, en, and ko rows in ai_enriched_poi.
            ai_poi_scope_df = (
                spark.table(POI_TABLE)
                    .filter(F.col('poi_id').isin(_enriched_poi_ids))
            )
            print(f'ai_enriched_poi merge: scoped to {len(_enriched_poi_ids)} enriched poi_id(s) (vi + en + ko)')

        # Explicit rename map (§3.7 of the design).
        REF_RENAME = {
            'description':           'ref_description',
            'time_to_spend_minutes': 'ref_time_to_spend_minutes',
            'best_time_type':        'ref_best_time_type',
            'best_time_from_month':  'ref_best_time_from_month',
            'best_time_to_month':    'ref_best_time_to_month',
            'cost_vnd':              'ref_cost_vnd',
            'cost_method':           'ref_cost_method',
            'cost_confidence':       'ref_cost_confidence',
            'experiences':           'ref_experiences',
            'review_summary':        'ref_review_summary',
            'reasoning_short':       'ref_reasoning_short',
        }
        PASS_THROUGH = ['enrichment_id', 'enrichment_job_id', 'enrichment_status',
                        'validation_status', 'model_name', 'source_updated_at', 'enriched_at']

        enrichment_select = enrichment_scope_df.select(
            F.col('poi_id'),
            *[F.col(src).alias(dst) for src, dst in REF_RENAME.items()],
            *[F.col(c) for c in PASS_THROUGH],
            # Override sources for the ai_poi categorical columns. Per docs/merge_script.sql
            # (lines 180-181, 198-199), ai_enriched_poi.business_sector / .business_category
            # come from the LLM's ref_business_sector / ref_type — NOT from ai_poi.
            # Map label -> canonical enum (per "Mapping data trên KB"); coalesce falls back
            # to the raw value if it's already an enum (idempotent) or unmapped (surfaces
            # as-is for QA rather than silently nulling it).
            F.coalesce(_sector_map_expr[F.col('ref_business_sector')], F.col('ref_business_sector'))
                .alias('_ref_business_sector'),
            F.coalesce(_cat_map_expr[F.col('ref_type')], F.col('ref_type'))
                .alias('_ref_business_category'),
        )

        # Diagnostic: flag any ref_business_sector / ref_type value that didn't match either
        # enum dict AND wasn't already a valid enum value — i.e. genuinely unmapped labels
        # that need a doc update or LLM prompt fix. Scoped to _enriched_poi_ids so a
        # partner_portal run only warns about rows it's actually touching, not the whole table.
        _known_sector_enums = set(BUSINESS_SECTOR_LABEL_TO_ENUM.values())
        _known_category_enums = set(CATEGORY_LABEL_TO_ENUM.values())
        _unmapped = (
            enrichment_select
            .filter(F.col('poi_id').isin(_enriched_poi_ids))
            .where(
                (~F.col('_ref_business_sector').isin(list(_known_sector_enums)) & F.col('_ref_business_sector').isNotNull())
                | (~F.col('_ref_business_category').isin(list(_known_category_enums)) & F.col('_ref_business_category').isNotNull())
            )
            .select('poi_id', '_ref_business_sector', '_ref_business_category')
        )
        _n_unmapped = _unmapped.count()
        if _n_unmapped > 0:
            print(f'WARNING: {_n_unmapped} row(s) have business_sector/category values not found in the enum map:')
            _unmapped.show(20, truncate=False)

        # If the target table already exists, conform to its exact schema; otherwise let Spark
        # infer the schema from the joined frame on first write.
        target_exists = spark.catalog.tableExists(TARGET_AI_ENRICHED_POI_OUTPUT)
        if target_exists:
            target_schema = spark.table(TARGET_AI_ENRICHED_POI_OUTPUT).schema
            target_cols = [f.name for f in target_schema.fields]
            # Drop ai_poi columns that aren't part of ai_enriched_poi (the 21 amenity columns).
            ai_poi_keep_cols = [c for c in ai_poi_scope_df.columns if c in target_cols]
        else:
            target_schema = None
            ai_poi_keep_cols = ai_poi_scope_df.columns  # keep all on first create
            print(f'  ai_enriched_poi table not found — will create from inferred schema on first write')

        ai_poi_select = ai_poi_scope_df.select(*ai_poi_keep_cols)

        # poi_vi / poi_en / poi_ko are the deterministic sibling poi_ids of the same physical place,
        # computed from (ingestion_source, source_id) per the project formula
        # uuid_8_4_4_4_12(md5(ingestion_source + source_id + lang)). NULL when source_id is
        # NULL (md5(concat(... NULL ...)) returns NULL in Spark).
        _UUID_PATTERN = r'(.{8})(.{4})(.{4})(.{4})(.{12})'

        def _sibling_poi_id_col(lang_literal):
            return F.regexp_replace(
                F.md5(F.concat(F.col('ingestion_source'), F.col('source_id'), F.lit(lang_literal))),
                _UUID_PATTERN, r'$1-$2-$3-$4-$5'
            )

        joined = ai_poi_select.join(enrichment_select, on='poi_id', how='left')

        if RUN_SCOPE == 'partner_portal':
            # Partner-portal POIs: keep original business_sector/business_category from ai_poi
            # when present, fall back to LLM ref_* only if the original is NULL. The *original*
            # value from POI_TABLE is also a raw partner-submitted label (not an enum), so it
            # must go through the same label->enum lookup before the coalesce, or coalesce just
            # preserves the unmapped label.
            joined = (
                joined
                .withColumn('_orig_business_sector_mapped',
                            F.coalesce(_sector_map_expr[F.col('business_sector')], F.col('business_sector')))
                .withColumn('_orig_business_category_mapped',
                            F.coalesce(_cat_map_expr[F.col('business_category')], F.col('business_category')))
                .withColumn('business_sector',
                            F.coalesce(F.col('_orig_business_sector_mapped'), F.col('_ref_business_sector')))
                .withColumn('business_category',
                            F.coalesce(F.col('_orig_business_category_mapped'), F.col('_ref_business_category')))
                .drop('_orig_business_sector_mapped', '_orig_business_category_mapped')
            )
            print("RUN_SCOPE='partner_portal': business_sector/category = original (enum-mapped)",
                "falling back to LLM ref_* (enum-mapped) if NULL (no row-level filter applied)")
        else:
            # Require ref_business_sector / ref_type to be non-NULL — matches the
            # WHERE filter in docs/merge_script.sql.
            joined = joined.where(
                F.col('_ref_business_sector').isNotNull()
                & F.col('_ref_business_category').isNotNull()
            )
            # Overwrite categorical columns with the LLM ref values (UPDATE SET behavior).
            joined = (
                joined
                .withColumn('business_sector',   F.col('_ref_business_sector'))
                .withColumn('business_category', F.col('_ref_business_category'))
            )

        joined = (
            joined
            # Add sibling identity columns.
            .withColumn('poi_vi', _sibling_poi_id_col('vi'))
            .withColumn('poi_en', _sibling_poi_id_col('en'))
            .withColumn('poi_ko', _sibling_poi_id_col('ko'))
            # updated_at = "last touched by anything" — overwrite ai_poi.updated_at with
            # GREATEST(ai_poi.updated_at, enriched_at). NULL-tolerant via F.greatest.
            .withColumn(
                'updated_at',
                F.greatest(F.col('updated_at'), F.col('enriched_at')),
            )
            .drop('_ref_business_sector', '_ref_business_category')
        )

        if target_schema is not None:
            # Cast to target schema, filling any missing target columns with NULLs.
            final_select = []
            for f in target_schema.fields:
                if f.name in joined.columns:
                    final_select.append(F.col(f.name).cast(f.dataType).alias(f.name))
                else:
                    final_select.append(F.lit(None).cast(f.dataType).alias(f.name))
            enriched_df = joined.select(*final_select)
        else:
            enriched_df = joined

        # Upsert by poi_id via Delta MERGE (DELETE … WHERE poi_id IN (SELECT …) is not
        # supported in this Fabric Delta runtime). MERGE handles update + insert in one pass.
        if target_exists:
            # Defensive de-dup within the batch so MERGE doesn't see two source rows for the
            # same target poi_id (Delta MERGE forbids that). Keep the freshest row by
            # enriched_at when available.
            from pyspark.sql.window import Window as _W
            if 'enriched_at' in enriched_df.columns:
                _dedup_w = _W.partitionBy('poi_id').orderBy(F.col('enriched_at').desc_nulls_last())
            else:
                _dedup_w = _W.partitionBy('poi_id').orderBy(F.lit(1))
            enriched_df = (
                enriched_df
                .withColumn('_rn', F.row_number().over(_dedup_w))
                .filter(F.col('_rn') == 1)
                .drop('_rn')
            )
            enriched_df.createOrReplaceTempView('incoming_ai_enriched_poi')
            spark.sql(f"""
                MERGE INTO {TARGET_AI_ENRICHED_POI_OUTPUT} AS target
                USING incoming_ai_enriched_poi AS source
                ON target.poi_id = source.poi_id
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
            """)
        else:
            enriched_df.write.mode('overwrite').saveAsTable(TARGET_AI_ENRICHED_POI_OUTPUT)

        spark.catalog.refreshTable(TARGET_AI_ENRICHED_POI_OUTPUT)
        n_written = enriched_df.count()
        print(f'ai_enriched_poi: upserted {n_written} row(s)')

        # Cleanup historical rows with NULL business_sector / business_category — from
        # before the ref_* override was wired in. Simple non-subquery DELETE; Delta-supported.
        #
        # RUN_SCOPE='partner_portal': this must NOT be a full-table cleanup — it should only
        # ever touch the partner-portal poi_ids this run scoped above (_enriched_poi_ids),
        # so a partner_portal run never deletes rows belonging to other ingestion
        # sources/scopes. Delta on this runtime doesn't support a correlated-subquery
        # DELETE (`poi_id IN (SELECT ...)`), so the scoped id list is inlined as SQL
        # literals instead.
        if RUN_SCOPE == 'partner_portal':
            if _enriched_poi_ids:
                _scope_ids_sql = ', '.join(
                    "'" + pid.replace("'", "''") + "'" for pid in _enriched_poi_ids
                ) 
                spark.sql(f"""
                    DELETE FROM {TARGET_AI_ENRICHED_POI_OUTPUT}
                    WHERE (business_sector IS NULL OR business_category IS NULL)
                    AND poi_id IN ({_scope_ids_sql})
                """)
            else:
                print('ai_enriched_poi cleanup: skipped — no partner-portal poi_id(s) in scope.')
        else:
            spark.sql(f"""
                DELETE FROM {TARGET_AI_ENRICHED_POI_OUTPUT}
                WHERE business_sector IS NULL OR business_category IS NULL
            """)
        spark.catalog.refreshTable(TARGET_AI_ENRICHED_POI_OUTPUT)
        _remaining_nulls = (
            spark.table(TARGET_AI_ENRICHED_POI_OUTPUT)
                .filter(F.col('business_sector').isNull() | F.col('business_category').isNull())
                .count()
        )
        print(f'ai_enriched_poi cleanup: rows with NULL business_sector/category remaining = {_remaining_nulls}')


StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 18, Finished, Available, Finished, False)

RUN_SCOPE='partner_portal': restricted ai_enriched_poi merge to 244 partner-portal poi_id(s) (no full merge)
destination_id filter: 244 -> 204 poi_id(s) after excluding NULL destination_id
ai_enriched_poi merge: scoped to 204 enriched poi_id(s) (vi + en + ko)
RUN_SCOPE='partner_portal': business_sector/category = original (enum-mapped) falling back to LLM ref_* (enum-mapped) if NULL (no row-level filter applied)
ai_enriched_poi: upserted 204 row(s)
ai_enriched_poi cleanup: rows with NULL business_sector/category remaining = 0


## 4.6. Display formatting

In [17]:
poi_prod = spark.table(TARGET_POI_OUTPUT)
poi_scope = poi_prod.filter(F.col('is_current') == True)

# Limit to rows that have been enriched in production
poi_scope = poi_prod.filter(F.col('is_current').isNotNull())

print(f"Sample of {TARGET_POI_OUTPUT}")
display(poi_scope.orderBy(F.col("enriched_at").desc_nulls_last()).limit(50))

StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 19, Finished, Available, Finished, False)

Sample of poi_enrichment


SynapseWidget(Synapse.DataFrame, 0c58a8ad-4829-4e29-ba5a-813453421e70)

In [18]:
# Display field labels per (display_category, language).

DISPLAY_FIELD_KEYS = {
    'accommodation': {
        'time_to_spend': {'vi': 'Thời gian lưu trú lý tưởng', 'en': 'Ideal stay duration', 'ko': '권장 숙박 시간'},
        'best_time': {'vi': 'Mùa lưu trú đẹp nhất', 'en': 'Best stay season', 'ko': '추천 숙박 시즌'},
        'cost': {'vi': 'Giá phòng tham khảo', 'en': 'Reference room rate', 'ko': '참고 객실 요금'},
        'operating_hours': {'vi': 'Giờ hoạt động', 'en': 'Operating hours', 'ko': '운영 시간'},
    },
    'accomodation': {
        'time_to_spend': {'vi': 'Thời gian lưu trú lý tưởng', 'en': 'Ideal stay duration', 'ko': '권장 숙박 시간'},
        'best_time': {'vi': 'Mùa lưu trú đẹp nhất', 'en': 'Best stay season', 'ko': '추천 숙박 시즌'},
        'cost': {'vi': 'Giá phòng tham khảo', 'en': 'Reference room rate', 'ko': '참고 객실 요금'},
        'operating_hours': {'vi': 'Giờ hoạt động', 'en': 'Operating hours', 'ko': '운영 시간'},
    },
    'f_n_b': {
        'time_to_spend': {'vi': 'Thời gian dùng bữa', 'en': 'Meal duration', 'ko': '식사 소요 시간'},
        'best_time': {'vi': 'Thời điểm nên ghé', 'en': 'Best time to visit', 'ko': '방문하기 좋은 시간'},
        'cost': {'vi': 'Mức giá tham khảo', 'en': 'Reference price', 'ko': '참고 가격'},
        'operating_hours': {'vi': 'Giờ phục vụ', 'en': 'Operating hours', 'ko': '영업 시간'},
    },
    'attraction': {
        'time_to_spend': {'vi': 'Thời gian tham quan', 'en': 'Visit duration', 'ko': '관람 소요 시간'},
        'best_time': {'vi': 'Mùa đẹp nhất', 'en': 'Best season', 'ko': '추천 시즌'},
        'cost': {'vi': 'Chi phí tham quan', 'en': 'Visit cost', 'ko': '관람 비용'},
        'operating_hours': {'vi': 'Giờ tham quan', 'en': 'Operating hours', 'ko': '관람 시간'},
    },
    'tour_n_activities': {
        'time_to_spend': {'vi': 'Thời lượng hoạt động', 'en': 'Activity duration', 'ko': '활동 소요 시간'},
        'best_time': {'vi': 'Thời điểm phù hợp', 'en': 'Suitable time', 'ko': '적합한 시기'},
        'cost': {'vi': 'Chi phí tham khảo', 'en': 'Reference cost', 'ko': '참고 비용'},
        'operating_hours': {'vi': 'Giờ hoạt động', 'en': 'Operating hours', 'ko': '운영 시간'},
    },
    'shopping_n_retail': {
        'time_to_spend': {'vi': 'Thời gian mua sắm', 'en': 'Shopping duration', 'ko': '쇼핑 소요 시간'},
        'best_time': {'vi': 'Thời điểm nên ghé', 'en': 'Best time to visit', 'ko': '방문하기 좋은 시간'},
        'cost': {'vi': 'Chi tiêu tham khảo', 'en': 'Reference spend', 'ko': '참고 지출'},
        'operating_hours': {'vi': 'Giờ mở cửa', 'en': 'Operating hours', 'ko': '영업 시간'},
    },
    'transport_route_base': {
        'time_to_spend': {'vi': 'Thời gian đưa đón', 'en': 'Transfer duration', 'ko': '이동 소요 시간'},
        'best_time': {'vi': 'Thời điểm phù hợp', 'en': 'Suitable time', 'ko': '적합한 시간'},
        'cost': {'vi': 'Chi phí đưa đón', 'en': 'Transfer cost', 'ko': '이동 비용'},
        'operating_hours': {'vi': 'Giờ hoạt động', 'en': 'Operating hours', 'ko': '운영 시간'},
    },
    'transport_on_demand': {
        'time_to_spend': {'vi': 'Thời gian thuê phổ biến', 'en': 'Common rental duration', 'ko': '일반 대여 시간'},
        'best_time': {'vi': 'Thời điểm nên thuê', 'en': 'Best rental time', 'ko': '대여하기 좋은 시간'},
        'cost': {'vi': 'Giá thuê tham khảo', 'en': 'Reference rental rate', 'ko': '참고 대여 요금'},
        'operating_hours': {'vi': 'Giờ hoạt động', 'en': 'Operating hours', 'ko': '운영 시간'},
    },
    'vehicle_rental': {
        'time_to_spend': {'vi': 'Thời gian thuê phổ biến', 'en': 'Common rental duration', 'ko': '일반 대여 시간'},
        'best_time': {'vi': 'Thời điểm nên thuê', 'en': 'Best rental time', 'ko': '대여하기 좋은 시간'},
        'cost': {'vi': 'Giá thuê tham khảo', 'en': 'Reference rental rate', 'ko': '참고 대여 요금'},
        'operating_hours': {'vi': 'Giờ phục vụ', 'en': 'Operating hours', 'ko': '영업 시간'},
    },
    'spa_n_wellness': {
        'time_to_spend': {'vi': 'Thời lượng liệu trình', 'en': 'Treatment duration', 'ko': '관리 소요 시간'},
        'best_time': {'vi': 'Thời điểm phù hợp', 'en': 'Suitable time', 'ko': '적합한 시간'},
        'cost': {'vi': 'Giá dịch vụ tham khảo', 'en': 'Reference service rate', 'ko': '참고 서비스 요금'},
        'operating_hours': {'vi': 'Giờ phục vụ', 'en': 'Operating hours', 'ko': '영업 시간'},
    },
    'event': {
        'time_to_spend': {'vi': 'Thời lượng chương trình', 'en': 'Event duration', 'ko': '행사 소요 시간'},
        'best_time': {'vi': 'Thời điểm nên xem', 'en': 'Best time to attend', 'ko': '참석하기 좋은 시간'},
        'cost': {'vi': 'Giá vé tham khảo', 'en': 'Reference ticket price', 'ko': '참고 티켓 가격'},
        'operating_hours': {'vi': 'Khung giờ diễn ra', 'en': 'Operating hours', 'ko': '진행 시간'},
    },
    'other': {
        'time_to_spend': {'vi': 'Thời lượng trải nghiệm', 'en': 'Experience duration', 'ko': '체험 소요 시간'},
        'best_time': {'vi': 'Thời điểm phù hợp', 'en': 'Best time', 'ko': '적합한 시간'},
        'cost': {'vi': 'Chi phí tham khảo', 'en': 'Reference cost', 'ko': '참고 비용'},
        'operating_hours': {'vi': 'Giờ hoạt động', 'en': 'Operating hours', 'ko': '운영 시간'},
    },
}

POI_DAY_LABELS = {
    'vi': ['Chủ nhật', 'Thứ 2', 'Thứ 3', 'Thứ 4', 'Thứ 5', 'Thứ 6', 'Thứ 7'],
    'en': ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday'],
    'ko': ['일요일', '월요일', '화요일', '수요일', '목요일', '금요일', '토요일'],
}

POI_OPERATING_HOURS_ALL_DAY = {
    'vi': 'Mở cả ngày',
    'en': 'Open all day',
    'ko': '종일 운영',
}

POI_OPERATING_HOURS_OPEN_NOW = {
    'vi': 'Đang mở',
    'en': 'Open',
    'ko': '영업 중',
}

MONTH_NAMES_EN = {
    1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June',
    7: 'July', 8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December',
}
MONTH_NAMES_KO = {month: f'{month}월' for month in range(1, 13)}


def infer_display_category(row):
    sector = str(row.get('business_sector') or '').strip().lower()
    return sector if sector in DISPLAY_FIELD_KEYS else 'other'


def display_key(display_category, field_name, lang):
    keys = DISPLAY_FIELD_KEYS.get(display_category, DISPLAY_FIELD_KEYS['other'])
    return keys[field_name].get(lang, keys[field_name]['en'])


def _plural_en(value, unit):
    return f'{value} {unit}' if value == 1 else f'{value} {unit}s'


def format_duration_display(minutes, display_category, lang):
    minutes = safe_int(minutes)
    if minutes is None or minutes <= 0:
        return 'Không cố định' if lang == 'vi' else '유동적' if lang == 'ko' else 'Flexible'

    if display_category in {'hotel', 'accommodation', 'accomodation'}:
        days = max(1, math.ceil(minutes / 1440))
        return f'{days} ngày' if lang == 'vi' else f'{days}일' if lang == 'ko' else _plural_en(days, 'day')

    if display_category in {'tour_activity', 'tour_n_activities'} and minutes >= 420:
        return 'Cả ngày' if lang == 'vi' else '종일' if lang == 'ko' else 'Full day'

    if minutes < 60:
        return f'{minutes} phút' if lang == 'vi' else f'{minutes}분' if lang == 'ko' else _plural_en(minutes, 'minute')

    hours = minutes // 60
    remaining = minutes % 60
    if remaining == 0:
        return f'{hours} tiếng' if lang == 'vi' else f'{hours}시간' if lang == 'ko' else _plural_en(hours, 'hour')
    if lang == 'vi':
        return f'{hours} tiếng {remaining} phút'
    if lang == 'ko':
        return f'{hours}시간 {remaining}분'
    return f'{_plural_en(hours, "hour")} {_plural_en(remaining, "minute")}'


def format_best_time_display(best_time_type, from_month, to_month, lang):
    start = safe_month(from_month)
    end = safe_month(to_month)
    if best_time_type != 'month_range' or start is None or end is None:
        return 'Quanh năm' if lang == 'vi' else '연중' if lang == 'ko' else 'Year round'
    if lang == 'vi':
        return f'Tháng {start} - Tháng {end}'
    if lang == 'ko':
        return f'{MONTH_NAMES_KO[start]} - {MONTH_NAMES_KO[end]}'
    return f'{MONTH_NAMES_EN[start]} - {MONTH_NAMES_EN[end]}'


def format_cost_display(cost_vnd, lang):
    amount = safe_int(cost_vnd)
    if amount is None:
        return None
    formatted = f'{round_vnd(amount):,.0f}'
    return f'Từ {formatted}đ' if lang == 'vi' else f'{formatted}동부터' if lang == 'ko' else f'From {formatted} vnd'

StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 20, Finished, Available, Finished, False)

In [19]:
# --- §4.6 (v5) Reconcile poi_enrichment_display against ai_poi as the source of
# truth — all-Spark, no toPandas() on the row payload (avoids numpy/pickle failures
# on shuffle). ai_poi is authoritative: this cell converges poi_enrichment_display to
# exactly that set (add missing, refresh stale, remove no-longer-active).
#
# v6: _AP_OVERRIDE_COLS is the single place that decides which *real* ai_poi columns
# replace their poi_enrichment counterparts. business_sector/ingestion_source always
# do; for RUN_SCOPE == 'partner_portal', description does too.
#
# display_category is NOT a column on ai_poi — for partner_portal it IS
# business_sector (used as-is, not inferred); for every other scope it's still
# inferred from business_sector via infer_display_category_udf. That's handled
# separately below, once reconciled_df already has business_sector on it.

from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, BooleanType, MapType
)
from pyspark.sql import functions as F



if len(poi_pd) > 0:

    _AP_OVERRIDE_COLS = {'business_sector', 'ingestion_source'}
    if RUN_SCOPE == 'partner_portal':
        _AP_OVERRIDE_COLS |= {'description'}

    # 1) ai_poi is the unique/authoritative set for this scope.
    ai_poi_base_df = (
        spark.table(POI_TABLE)
            .select('poi_id', 'lang', 'is_active', *_AP_OVERRIDE_COLS)
            .filter(F.col('is_active') == True)
            .dropDuplicates(['poi_id'])
    )
    if RUN_SCOPE == 'google':
        ai_poi_base_df = ai_poi_base_df.filter(F.col('ingestion_source') == 'google')
    elif RUN_SCOPE == 'partner_portal':
        ai_poi_base_df = ai_poi_base_df.filter(F.col('ingestion_source') == 'partner_portal')

    print(f'ai_poi_base_df: {ai_poi_base_df.count():,} distinct, active poi_id(s) in scope — this is the authoritative set')

    # 2) poi_enrichment content, deduped defensively (is_current not guaranteed unique
    #    per poi_id — flip-then-append write is two non-atomic steps).
    _current_w = Window.partitionBy('poi_id').orderBy(F.col('enriched_at').desc_nulls_last())
    poi_enrichment_src_df = (
        spark.table(POI_ENRICHMENT_TABLE)
            .filter((F.col('enrichment_status') == 'success') & (F.col('is_current') == True))
            .withColumn('_rn', F.row_number().over(_current_w))
            .filter(F.col('_rn') == 1)
            .drop('_rn')
    )

    # 3) Inner join against ai_poi_base_df — row set can never exceed the authoritative
    #    scope; dedup above guarantees no fan-out. pe.* drops _AP_OVERRIDE_COLS so it
    #    can't collide with the ap.* versions we're keeping instead.
    _pe_cols = [c for c in poi_enrichment_src_df.columns if c not in _AP_OVERRIDE_COLS]

    reconciled_df = (
        poi_enrichment_src_df.alias('pe')
        .join(ai_poi_base_df.alias('ap'), on='poi_id', how='inner')
        .select(
            *[F.col(f'pe.{c}') for c in _pe_cols],
            *[F.col(f'ap.{c}').alias(c) for c in _AP_OVERRIDE_COLS],
        )
    )
    _n_reconciled = reconciled_df.count()
    print(f'Reconciled {_n_reconciled:,} successful, current, active poi_enrichment row(s) against ai_poi')

    if _n_reconciled > 0:

        # --- formatting logic wrapped as UDFs (reuse existing python functions as-is) ---
        infer_display_category_udf = F.udf(
            lambda sector: infer_display_category({'business_sector': sector}), StringType()
        )
        duration_udf  = F.udf(format_duration_display, StringType())
        best_time_udf = F.udf(format_best_time_display, StringType())
        cost_udf      = F.udf(format_cost_display, StringType())

        @F.udf(StringType())
        def time_to_spend_json_udf(category, lang, text):
            key = display_key(category, 'time_to_spend', lang)
            return json.dumps({key: text}, ensure_ascii=False)

        @F.udf(StringType())
        def best_time_json_udf(category, lang, text):
            key = display_key(category, 'best_time', lang)
            return json.dumps({key: text}, ensure_ascii=False)

        @F.udf(StringType())
        def cost_json_udf(category, lang, text):
            if text is None:
                return json.dumps(None, ensure_ascii=False)
            key = display_key(category, 'cost', lang)
            return json.dumps({key: text}, ensure_ascii=False)

        display_generated_at = current_ts.to_pydatetime()

        # display_category: partner_portal has no such column on ai_poi — its
        # business_sector value (already on reconciled_df, sourced from ap above) IS
        # the display category, used as-is. Every other scope still infers it from
        # business_sector via the UDF, as before.
        if RUN_SCOPE == 'partner_portal':
            _display_category_expr = F.coalesce(F.col('business_sector'), F.lit('other'))
        else:
            _display_category_expr = F.coalesce(infer_display_category_udf('business_sector'), F.lit('other'))

        poi_enrichment_display_df = (
            reconciled_df
            .withColumn('lang', F.when(F.col('lang').isin(SUPPORTED_LANGS), F.col('lang')).otherwise(F.lit('en')))
            .withColumn('display_category', _display_category_expr)
            .withColumn('_duration_text', duration_udf('time_to_spend_minutes', 'display_category', 'lang'))
            .withColumn('_best_time_text', best_time_udf('best_time_type', 'best_time_from_month', 'best_time_to_month', 'lang'))
            .withColumn('_cost_text', cost_udf('cost_vnd', 'lang'))
            .withColumn('time_to_spend', time_to_spend_json_udf('display_category', 'lang', '_duration_text'))
            .withColumn('best_time', best_time_json_udf('display_category', 'lang', '_best_time_text'))
            .withColumn('cost', cost_json_udf('display_category', 'lang', '_cost_text'))
            .withColumn('display_id', F.concat(F.col('enrichment_id'), F.lit('_display')))
            .withColumn('display_generated_at', F.lit(display_generated_at).cast(TimestampType()))
            .withColumn('is_current', F.lit(True))
            .select(
                F.col('display_id'),
                F.col('enrichment_id'),
                F.col('poi_id'),
                F.col('poi_name'),
                F.col('lang'),
                F.col('display_category'),
                F.col('description'),
                F.col('time_to_spend'),
                F.col('best_time'),
                F.col('cost'),
                F.col('experiences').cast(StringType()).alias('experiences'),
                F.col('enrichment_job_id'),
                F.col('validation_status'),
                F.col('display_generated_at'),
                F.col('is_current'),
                F.col('ingestion_source'),
            )
        )

        poi_enrichment_display_schema = StructType([
            StructField('display_id',           StringType(),    False),
            StructField('enrichment_id',        StringType(),    False),
            StructField('poi_id',               StringType(),    False),
            StructField('poi_name',             StringType(),    True),
            StructField('lang',                 StringType(),    True),
            StructField('display_category',     StringType(),    True),
            StructField('description',          StringType(),    True),
            StructField('time_to_spend',        StringType(),    True),
            StructField('best_time',            StringType(),    True),
            StructField('cost',                 StringType(),    True),
            StructField('experiences',          StringType(),    True),
            StructField('enrichment_job_id',    StringType(),    True),
            StructField('validation_status',    StringType(),    True),
            StructField('display_generated_at', TimestampType(), True),
            StructField('is_current',           BooleanType(),   True),
            StructField('ingestion_source',     StringType(),    True),
        ])

        # Enforce final schema explicitly (fail fast on type mismatch instead of a
        # deferred pickle/shuffle failure).
        poi_enrichment_display_df = poi_enrichment_display_df.select(*[
            F.col(f.name).cast(f.dataType).alias(f.name) for f in poi_enrichment_display_schema.fields
        ])

        _dedup_w = Window.partitionBy('poi_id', 'lang').orderBy(F.col('display_generated_at').desc_nulls_last())
        poi_enrichment_display_df = (
            poi_enrichment_display_df
            .withColumn('_rn', F.row_number().over(_dedup_w))
            .filter(F.col('_rn') == 1)
            .drop('_rn')
        )

        print(f'Built {poi_enrichment_display_df.count():,} display row(s); schema has {len(poi_enrichment_display_schema.fields)} field(s).')

        def _conform_table_to_schema(table_name, desired_schema):
            live_types = {f.name: f.dataType for f in spark.table(table_name).schema.fields}
            if not any(f.name in live_types and live_types[f.name] != f.dataType
                    for f in desired_schema.fields):
                return
            proj = []
            for f in desired_schema.fields:
                if f.name not in live_types:
                    continue
                c = F.col(f.name)
                if isinstance(live_types[f.name], MapType) and isinstance(f.dataType, StringType):
                    c = F.to_json(c)
                proj.append(c.cast(f.dataType).alias(f.name))
            migrated = spark.table(table_name).select(*proj)
            migrated.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(table_name)
            spark.catalog.refreshTable(table_name)

        if DRY_RUN_MODE:
            poi_enrichment_display_df.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(TARGET_POI_DISPLAY_OUTPUT)
        elif spark.catalog.tableExists(TARGET_POI_DISPLAY_OUTPUT):
            _conform_table_to_schema(TARGET_POI_DISPLAY_OUTPUT, poi_enrichment_display_schema)
            poi_enrichment_display_df.createOrReplaceTempView('incoming_poi_display')

            _upd_cols = [f.name for f in poi_enrichment_display_schema.fields]  # excludes created_at/updated_at, target-only

            _update_set = ', '.join(f'{c} = source.{c}' for c in _upd_cols)
            _update_set += ', updated_at = current_timestamp()'

            _insert_cols = _upd_cols + ['created_at', 'updated_at']
            _insert_vals = [f'source.{c}' for c in _upd_cols] + ['current_timestamp()', 'current_timestamp()']

            spark.sql(f"""
                MERGE INTO {TARGET_POI_DISPLAY_OUTPUT} AS target
                USING incoming_poi_display AS source
                ON target.poi_id = source.poi_id AND target.lang = source.lang
                WHEN MATCHED THEN UPDATE SET {_update_set}
                WHEN NOT MATCHED THEN INSERT ({', '.join(_insert_cols)}) VALUES ({', '.join(_insert_vals)})
            """)

    # 4) Enforce ai_poi as the unique/authoritative set: remove display rows for
    #    poi_id/lang pairs no longer active in ai_poi — scoped to this run's
    #    ingestion_source so a google run never touches partner_portal rows.
    if RUN_SCOPE in ('google', 'partner_portal'):
        _active_scope_df = (
            spark.table(POI_TABLE)
                    .select('poi_id', 'lang')
                    .filter((F.col('is_active') == True) & (F.col('ingestion_source') == RUN_SCOPE))
                    .distinct()
        )
        _active_scope_df.createOrReplaceTempView('active_ai_poi_scope')

        spark.sql(f"""
            MERGE INTO {TARGET_POI_DISPLAY_OUTPUT} AS target
            USING active_ai_poi_scope AS source
            ON target.poi_id = source.poi_id AND target.lang = source.lang
            WHEN NOT MATCHED BY SOURCE AND target.ingestion_source = '{RUN_SCOPE}'
                THEN DELETE
        """)
    else:
        print(f"RUN_SCOPE={RUN_SCOPE!r}: skipping the authoritative-set delete "
                f"(only wired up for 'google' and 'partner_portal' scopes)")


StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 21, Finished, Available, Finished, False)

ai_poi_base_df: 356 distinct, active poi_id(s) in scope — this is the authoritative set
Reconciled 234 successful, current, active poi_enrichment row(s) against ai_poi
Built 234 display row(s); schema has 16 field(s).


### Add operating hours

In [20]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
if len(poi_pd)>0:


    target = DeltaTable.forName(spark, "ai_enriched_poi")

    # ── Step 0: clean out current operating_hours for partner_portal rows before merging ──
    target.update(
        condition="ingestion_source = 'partner_portal'",
        set={"operating_hours": "NULL"}
    )

    # ── Step 1: parse poi_detail's operating hours JSON (no ingestion_source filter here) ──
    poi_detail_df = spark.table("poi_detail")

    schema = "STRUCT<timezone:STRING, is_24_7:BOOLEAN, periods_json:ARRAY<STRUCT<day_of_week:INT,open_hour:INT,open_minute:INT,close_hour:INT,close_minute:INT>>, holiday_exceptions_json:ARRAY<STRING>>"

    parsed_df = (
        poi_detail_df
        .filter(F.col("poi_opening_hours_json").isNotNull())
        .withColumn("oh", F.from_json(F.col("poi_opening_hours_json"), schema))
    )

    source_df = parsed_df.withColumn(
        "new_operating_hours",
        F.when(
            F.col("oh.is_24_7") == True,
            F.to_json(F.array(F.struct(
                F.lit(None).cast("int").alias("close_hour"),
                F.lit(None).cast("int").alias("close_day"),
                F.lit(0).alias("open_hour"),
                F.lit(None).cast("int").alias("close_minute"),
                F.lit(0).alias("open_minute"),
                F.lit(0).alias("day_of_week"),
            )))
        ).when(
            F.col("oh.periods_json").isNull() | (F.size(F.col("oh.periods_json")) == 0),
            F.lit(None)
        ).otherwise(
            F.to_json(F.transform(
                F.col("oh.periods_json"),
                lambda p: F.struct(
                    p["close_hour"].alias("close_hour"),
                    F.when(p["close_hour"].isNull(), F.lit(None).cast("int"))
                    .when(
                        (p["close_hour"] < p["open_hour"]) |
                        ((p["close_hour"] == p["open_hour"]) & (p["close_minute"] < p["open_minute"])),
                        (p["day_of_week"] + 1) % 7
                    )
                    .otherwise(p["day_of_week"])
                    .alias("close_day"),
                    p["open_hour"].alias("open_hour"),
                    p["close_minute"].alias("close_minute"),
                    p["open_minute"].alias("open_minute"),
                    p["day_of_week"].alias("day_of_week"),
                )
            ))
        )
    ).select("poi_id", "new_operating_hours")

    # ── Step 2: merge into ai_enriched_poi, scoped to partner_portal rows only ──
    (
        target.alias("t")
        .merge(
            source_df.alias("s"),
            "t.poi_id = s.poi_id AND t.ingestion_source = 'partner_portal'"
        )
        .whenMatchedUpdate(set={"operating_hours": "s.new_operating_hours"})
        .execute()
    )

StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 22, Finished, Available, Finished, False)

In [21]:
%%sql
select * from ai_enriched_poi where ingestion_source = 'partner_portal'

StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 23, Finished, Available, Finished, False)

<Spark SQL result set with 209 rows and 107 fields>

#### Deactive google poi that link to partner_poi

In [22]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ---- CONFIG ----
AI_POI_TABLE = "lh_vv_gold.dbo.ai_poi"
BSPL_TABLE = "lh_vv_bronze.partner.business_service_poi_link"   # e.g. "lh_vv_bronze.partner.business_service_poi_link"
# -----------------



bspl = spark.table(BSPL_TABLE).filter(F.col("deleted") == 0).alias("bspl")
ai_poi = spark.table(AI_POI_TABLE)

google_ap = ai_poi.filter(F.col("ingestion_source") == "google").alias("google_ap")
partner_ap = ai_poi.filter(F.col("ingestion_source") == "partner_portal").alias("partner_ap")

# resolve every bspl row to the google source_id it points at 
bspl_google_source = (
    bspl.join(google_ap, F.col("bspl.poi_id") == F.col("google_ap.poi_id"), "inner")
        .select(
            F.col("bspl.business_service_id").alias("business_service_id"),
            F.col("google_ap.source_id").alias("google_source_id"),
        )
        .distinct()
        .alias("bgs")
)

google_ap2 = ai_poi.filter(F.col("ingestion_source") == "google").alias("google_ap2")

# fan out to both language siblings of EVERY google source_id linked, however many there are
candidate_google_pois = (
    bspl_google_source.join(
        google_ap2, F.col("bgs.google_source_id") == F.col("google_ap2.source_id"), "inner"
    )
    .select(
        F.col("bgs.business_service_id").alias("business_service_id"),
        F.col("google_ap2.poi_id").alias("google_poi_id"),
        F.col("google_ap2.poi_name").alias("google_poi_name"),
    )
    .distinct()
)

ai_enriched_poi = spark.table(AI_ENRICHED_POI_TABLE)

google_enriched_ids = (
    ai_enriched_poi.filter(F.col("ingestion_source") == "google")
    .select(F.col("poi_id").alias("enriched_poi_id"))
    .distinct()
)
partner_enriched_source_ids = (
    ai_enriched_poi.filter(F.col("ingestion_source") == "partner_portal")
    .select(F.col("source_id").alias("business_service_id"))
    .distinct()
)

google_pois_to_deactivate = (
    candidate_google_pois
    .join(
        google_enriched_ids,
        candidate_google_pois["google_poi_id"] == google_enriched_ids["enriched_poi_id"],
        "inner",
    )
    .join(partner_enriched_source_ids, on="business_service_id", how="inner")
    .select("business_service_id", "google_poi_id", "google_poi_name")
    .distinct()
    .cache()
)

# one partner-portal display name per business_service_id (prefer 'en', fallback to any)
partner_names = (
    partner_ap.select(
        F.col("partner_ap.source_id").alias("business_service_id"),
        F.col("partner_ap.poi_name").alias("partner_poi_name"),
        F.col("partner_ap.lang").alias("lang"),
    )
    .withColumn("lang_rank", F.when(F.col("lang") == "en", 0).when(F.col("lang") == "vi", 1).when(F.col("lang") == "ko", 2).otherwise(3))
    .withColumn(
        "rn",
        F.row_number().over(Window.partitionBy("business_service_id").orderBy("lang_rank")),
    )
    .filter(F.col("rn") == 1)
    .select("business_service_id", "partner_poi_name")
)

report = (
    google_pois_to_deactivate
    .join(partner_names, on="business_service_id", how="left")
    .select("business_service_id", "partner_poi_name", "google_poi_id", "google_poi_name")
    .distinct()
    .collect()
)

target_ids = [r.google_poi_id for r in report]
print(f"{len(target_ids)} google poi_id(s) qualify for deactivation")

if target_ids:
    to_merge = google_pois_to_deactivate.select("google_poi_id").distinct()

    dt_enriched = DeltaTable.forName(spark, AI_ENRICHED_POI_TABLE)
    (
        dt_enriched.alias("tgt")
        .merge(to_merge.alias("src"), "tgt.poi_id = src.google_poi_id AND tgt.ingestion_source = 'google'")
        .whenMatchedUpdate(set={"is_active": F.lit(False)})
        .execute()
    )

    dt_master = DeltaTable.forName(spark, POI_MASTER_TABLE)
    (
        dt_master.alias("tgt")
        .merge(to_merge.alias("src"), "tgt.poi_id = src.google_poi_id AND tgt.ingestion_source = 'google'")
        .whenMatchedUpdate(set={"is_active": F.lit(False)})
        .execute()
    )

    print("Deactivated google poi_id(s):")
    for r in report:
        print(f"  - business: {r.partner_poi_name!r} ({r.business_service_id})")
        print(f"    google poi deactivated: {r.google_poi_id}  name: {r.google_poi_name!r}")
else:
    print("Nothing to deactivate — no google poi_id met both gating conditions.")


StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 24, Finished, Available, Finished, False)

0 google poi_id(s) qualify for deactivation
Nothing to deactivate — no google poi_id met both gating conditions.


### Post clean up by SQL
This will trigger when run, or can be run indepedantly in the notebook. Soft-delete the non-valid.

In [23]:
%%sql
SELECT * FROM poi_enrichment_display
WHERE description IS NULL OR LTRIM(RTRIM(description)) = ''
   OR cost        IS NULL OR LTRIM(RTRIM(cost)) IN ('', 'null', '[]', '{}')
   OR experiences IS NULL OR LTRIM(RTRIM(experiences)) IN ('', '[]', '{}');

StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 25, Finished, Available, Finished, False)

<Spark SQL result set with 64 rows and 18 fields>

In [24]:
%%sql
-- clean invalid enrichment display
DELETE FROM poi_enrichment_display
WHERE description IS NULL OR LTRIM(RTRIM(description)) = ''
   OR cost        IS NULL OR LTRIM(RTRIM(cost)) IN ('', 'null', '[]', '{}')
   OR experiences IS NULL OR LTRIM(RTRIM(experiences)) IN ('', '[]', '{}');

StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 26, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [25]:
%%sql
-- 1) Soft-delete ai_enriched_poi rows (partner_portal only) that have no matching display row
MERGE INTO ai_enriched_poi AS tgt
USING (
  SELECT DISTINCT poi_id, lang
  FROM poi_enrichment_display
) AS src
ON tgt.poi_id = src.poi_id AND tgt.lang = src.lang
WHEN NOT MATCHED BY SOURCE AND ingestion_source = 'partner_portal'
  THEN UPDATE SET is_active = false

StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 27, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [26]:
%%sql

-- 2) Balance ai_enriched_poi across languages (vi/en/ko) 
MERGE INTO ai_enriched_poi AS tgt
USING (
  SELECT source_id
  FROM ai_enriched_poi
  WHERE ingestion_source = 'partner_portal'
  GROUP BY source_id
  HAVING COUNT(DISTINCT lang) < 3
      OR MIN(CASE WHEN is_active THEN 1 ELSE 0 END) = 0
) AS src
ON tgt.source_id = src.source_id
WHEN MATCHED AND lower(trim(tgt.ingestion_source)) = 'partner_portal'
  THEN UPDATE SET is_active = false


StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 28, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [27]:
%%sql
-- 3) Sync poi_master (partner_portal only)
MERGE INTO poi_master AS tgt
USING (
  SELECT poi_id, lang,
         MAX(CASE WHEN is_active THEN 1 ELSE 0 END) = 1 AS src_active
  FROM ai_enriched_poi
  WHERE ingestion_source = 'partner_portal'
  GROUP BY poi_id, lang
) AS src
ON tgt.poi_id = src.poi_id AND tgt.lang = src.lang
WHEN MATCHED THEN UPDATE SET is_active = src.src_active
WHEN NOT MATCHED BY SOURCE AND tgt.ingestion_source = 'partner_portal' THEN UPDATE SET is_active = false

StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 29, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [28]:
%%sql
-- 4) Final check
SELECT 'ai_poi_vi_active' AS metric, count(*) AS cnt FROM ai_poi
WHERE lang = 'vi' AND is_active = 1 AND ingestion_source = 'partner_portal'
UNION ALL
SELECT 'ai_poi_en_active', count(*) FROM ai_poi
WHERE lang = 'en' AND is_active = 1 AND ingestion_source = 'partner_portal'
UNION ALL
SELECT 'ai_poi_ko_active', count(*) FROM ai_poi
WHERE lang = 'ko' AND is_active = 1 AND ingestion_source = 'partner_portal'
UNION ALL
SELECT 'enrich_vi_active', count(*) FROM ai_enriched_poi
WHERE lang = 'vi' AND is_active = 1 AND ingestion_source = 'partner_portal'
UNION ALL
SELECT 'enrich_en_active', count(*) FROM ai_enriched_poi
WHERE lang = 'en' AND is_active = 1 AND ingestion_source = 'partner_portal'
UNION ALL
SELECT 'enrich_ko_active', count(*) FROM ai_enriched_poi
WHERE lang = 'ko' AND is_active = 1 AND ingestion_source = 'partner_portal'
UNION ALL
SELECT 'master_vi_active', count(*) FROM poi_master
WHERE lang = 'vi' AND is_active = 1 AND ingestion_source = 'partner_portal'
UNION ALL
SELECT 'master_en_active', count(*) FROM poi_master
WHERE lang = 'en' AND is_active = 1 AND ingestion_source = 'partner_portal'
UNION ALL
SELECT 'master_ko_active', count(*) FROM poi_master
WHERE lang = 'ko' AND is_active = 1 AND ingestion_source = 'partner_portal'
ORDER BY metric;

StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 30, Finished, Available, Finished, False)

<Spark SQL result set with 9 rows and 2 fields>

#### Fall back for the case of too-long description in poi_master

In [29]:
# %% [markdown]
# ## Fallback — Trim poi_master.short_description + short_address exceeding 512 chars
# Safety net: some upstream sources occasionally write short_description /
# short_address longer than the 512-char column contract. Same word-boundary
# trim logic applied to both fields in a single MERGE.

# %%
from pyspark.sql.functions import (
    col, length as spark_length, substring, regexp_extract, when, concat, lit
)

MAX_LEN = 500
SUFFIX = '...'
CUT_LEN = MAX_LEN - len(SUFFIX)

def add_trimmed_col(df, src_col, out_col):
    return (
        df
        .withColumn(f'{src_col}_cut', substring(col(src_col), 1, CUT_LEN))
        .withColumn(f'{src_col}_last_space_chunk', regexp_extract(col(f'{src_col}_cut'), r'^(.*)\s', 1))
        .withColumn(
            out_col,
            when(spark_length(col(src_col)) <= MAX_LEN, col(src_col))
            .when(col(f'{src_col}_last_space_chunk') != '', concat(col(f'{src_col}_last_space_chunk'), lit(SUFFIX)))
            .otherwise(concat(col(f'{src_col}_cut'), lit(SUFFIX)))
        )
        .drop(f'{src_col}_cut', f'{src_col}_last_space_chunk')
    )

df_long_fields = (
    spark.table(POI_MASTER_TABLE)  # poi_master
    .filter(
        (spark_length(col('short_description')) > MAX_LEN) |
        (spark_length(col('short_address')) > MAX_LEN)
    )
    .select('poi_id', 'lang', 'short_description', 'short_address')
)

n_long = df_long_fields.count()
print(f"poi_master rows with short_description or short_address > {MAX_LEN} chars: {n_long}")

if n_long > 0:
    df_trimmed = add_trimmed_col(df_long_fields, 'short_description', 'short_description_trimmed')
    df_trimmed = add_trimmed_col(df_trimmed, 'short_address', 'short_address_trimmed')
    df_trimmed = df_trimmed.select('poi_id', 'lang', 'short_description_trimmed', 'short_address_trimmed')
    df_trimmed.createOrReplaceTempView('short_fields_trim_src')

    spark.sql(f"""
        MERGE INTO {POI_MASTER_TABLE} AS tgt
        USING short_fields_trim_src AS src
        ON tgt.poi_id = src.poi_id AND tgt.lang = src.lang
        WHEN MATCHED THEN UPDATE SET
            tgt.short_description = src.short_description_trimmed,
            tgt.short_address     = src.short_address_trimmed,
            tgt.updated_at        = current_timestamp()
    """)
    print(f"Trimmed {n_long} rows (short_description and/or short_address) in {POI_MASTER_TABLE}.")
else:
    print("No short_description or short_address values exceed the limit — nothing to trim.")


# %% [markdown]
# ## Fallback — Trim poi_enrichment_display.description exceeding 512 chars
# Same word-boundary trim logic as poi_master, applied to poi_enrichment_display.description.

# %%
df_long_desc = (
    spark.table(POI_ENRICHMENT_DISPLAY_TABLE)  # poi_enrichment_display
    .filter(spark_length(col('description')) > MAX_LEN)
    .select('poi_id', 'lang', 'description')
)

n_long_desc = df_long_desc.count()
print(f"poi_enrichment_display rows with description > {MAX_LEN} chars: {n_long_desc}")

if n_long_desc > 0:
    df_desc_trimmed = add_trimmed_col(df_long_desc, 'description', 'description_trimmed')
    df_desc_trimmed = df_desc_trimmed.select('poi_id', 'lang', 'description_trimmed')
    df_desc_trimmed.createOrReplaceTempView('description_trim_src')

    spark.sql(f"""
        MERGE INTO {POI_ENRICHMENT_DISPLAY_TABLE} AS tgt
        USING description_trim_src AS src
        ON tgt.poi_id = src.poi_id AND tgt.lang = src.lang
        WHEN MATCHED THEN UPDATE SET
            tgt.description  = src.description_trimmed,
            tgt.display_generated_at   = current_timestamp()
    """)
    print(f"Trimmed {n_long_desc} rows (description) in {POI_ENRICHMENT_DISPLAY_TABLE}.")
else:
    print("No description values exceed the limit — nothing to trim.")



StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 31, Finished, Available, Finished, False)

poi_master rows with short_description or short_address > 500 chars: 0
No short_description or short_address values exceed the limit — nothing to trim.
poi_enrichment_display rows with description > 500 chars: 0
No description values exceed the limit — nothing to trim.


In [30]:
%%sql
select * from ai_enriched_poi where ingestion_source = 'partner_portal' and is_active = True

StatementMeta(, 759b049e-3e14-42fa-8f01-e6473086bf31, 32, Finished, Available, Finished, False)

<Spark SQL result set with 105 rows and 107 fields>